# Train `Mjlab-Lift-Cube-SO101` on Google Colab GPU

This self-contained notebook installs [`mjlab`](https://github.com/mujocolab/mjlab), sets up the `so101-mjlab` package and SO-ARM101 meshes, trains a 50 Hz cube-lifting PPO policy on GPU, previews an evaluation rollout video, and downloads the exported `.onnx` policy for real-hardware deployment on your Mac (`uv run so101-deploy --onnx so101_lift_cube_policy.onnx`).

**Before running:** Make sure your Colab runtime is set to GPU (`Runtime -> Change runtime type -> T4 GPU` or `L4 GPU`).

In [ ]:
# 1. Verify GPU and install mjlab + ONNX export dependencies
!nvidia-smi
!pip install -q uv
!uv pip install --system "git+https://github.com/mujocolab/mjlab.git@main" onnx onnxscript onnxruntime pyserial mediapy

In [ ]:
import base64
import lzma
import urllib.request
from pathlib import Path

REPO_DIR = Path("/content/so101-mjlab")
ASSETS_DIR = REPO_DIR / "src/so101_mjlab/so101/xmls/assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

TEXT_FILES = {
  "pyproject.toml": '[project]\nname = "so101-mjlab"\nversion = "0.1.0"\ndescription = "SO-ARM101 manipulation environments and sim-to-real deployment for mjlab"\nreadme = "README.md"\nrequires-python = ">=3.10,<3.14"\ndependencies = [\n  "mjlab>=1.1.0",\n  "pyserial>=3.5",\n]\n\n[project.scripts]\nso101-deploy = "so101_mjlab.deploy_real:main"\n\n[build-system]\nrequires = ["uv_build>=0.8.18,<0.12.0"]\nbuild-backend = "uv_build"\n\n[dependency-groups]\ndev = [\n  "pytest>=8.0.0",\n  "ruff>=0.14.14",\n]\n\n[tool.ruff]\nindent-width = 2\nsrc = ["src"]\n\n[tool.ruff.lint]\nselect = ["E4", "E7", "E9", "F", "I", "B"]\n\n[project.entry-points."mjlab.tasks"]\nso101_mjlab = "so101_mjlab"\n',
  "README.md": "# so101-mjlab\n\nManipulation environments and 50 Hz sim-to-real deployment for the open-source 6-DoF **SO-ARM101** robot arm built on [`mjlab`](https://github.com/mujocolab/mjlab).\n\n## Registered Tasks\n\nInstalling `so101-mjlab` registers three manipulation tasks via the `mjlab.tasks` entry point:\n\n| Task ID | Observation Space | Description |\n| :--- | :--- | :--- |\n| `Mjlab-Lift-Cube-SO101` | 24-D state | Reach, grasp, and lift a 3 cm cube to randomized 3D targets using the SO-101 parallel jaw gripper. |\n| `Mjlab-Lift-Cube-SO101-Rgb` | State + 32x32 RGB (`wrist_cam`) | Visual cube lifting with domain-randomized cube colors and Spatial Softmax CNN policy. |\n| `Mjlab-Lift-Cube-SO101-Depth` | State + 32x32 Depth (`wrist_cam`) | Depth-based visual cube lifting from the wrist-mounted camera. |\n\n## Quick Start\n\n### 1. Preview the SO-101 Robot in MuJoCo\n\n```bash\nuv run python -m so101_mjlab.so101.so101_constants\n```\n\n### 2. Interactive Play & Evaluation\n\nEvaluate with a zero or dummy policy in the `mjlab` viewer:\n\n```bash\nuv run play Mjlab-Lift-Cube-SO101 --agent zero\n```\n\nEvaluate a trained checkpoint:\n\n```bash\nuv run play Mjlab-Lift-Cube-SO101 --checkpoint-file path/to/model.pt\n```\n\n### 3. Training (CUDA GPU / Colab)\n\nTrain the state-based policy with 4096 parallel environments:\n\n```bash\nuv run train Mjlab-Lift-Cube-SO101 --env.scene.num-envs 4096\n```\n\nTrain the wrist-camera RGB policy:\n\n```bash\nuv run train Mjlab-Lift-Cube-SO101-Rgb --env.scene.num-envs 2048\n```\n\n### 4. Physical SO-101 Hardware Deployment (macOS / Linux)\n\nDeploy an exported `.onnx` policy at 50 Hz over 1 Mbps Feetech STS3215 USB serial:\n\n```bash\n# Dry-run kinematic & ONNX verification without hardware plugged in\nuv run so101-deploy --dry-run\n\n# Live deployment on physical SO-101 follower arm\nuv run so101-deploy --onnx path/to/policy.onnx --port /dev/tty.usbmodem5B790143431\n```\n",
  "src/so101_mjlab/__init__.py": '"""SO-ARM101 manipulation tasks registered as an mjlab plugin package."""\n\nfrom mjlab.tasks.manipulation.rl import ManipulationOnPolicyRunner\nfrom mjlab.tasks.registry import register_mjlab_task\n\nfrom .env_cfgs import (\n  so101_lift_cube_env_cfg,\n  so101_lift_cube_vision_env_cfg,\n)\nfrom .rl_cfg import (\n  so101_lift_cube_ppo_runner_cfg,\n  so101_lift_cube_vision_ppo_runner_cfg,\n)\n\nregister_mjlab_task(\n  task_id="Mjlab-Lift-Cube-SO101",\n  env_cfg=so101_lift_cube_env_cfg(),\n  play_env_cfg=so101_lift_cube_env_cfg(play=True),\n  rl_cfg=so101_lift_cube_ppo_runner_cfg(),\n  runner_cls=ManipulationOnPolicyRunner,\n)\n\nregister_mjlab_task(\n  task_id="Mjlab-Lift-Cube-SO101-Rgb",\n  env_cfg=so101_lift_cube_vision_env_cfg(cam_type="rgb"),\n  play_env_cfg=so101_lift_cube_vision_env_cfg(cam_type="rgb", play=True),\n  rl_cfg=so101_lift_cube_vision_ppo_runner_cfg(),\n  runner_cls=ManipulationOnPolicyRunner,\n)\n\nregister_mjlab_task(\n  task_id="Mjlab-Lift-Cube-SO101-Depth",\n  env_cfg=so101_lift_cube_vision_env_cfg(cam_type="depth"),\n  play_env_cfg=so101_lift_cube_vision_env_cfg(cam_type="depth", play=True),\n  rl_cfg=so101_lift_cube_vision_ppo_runner_cfg(),\n  runner_cls=ManipulationOnPolicyRunner,\n)\n',
  "src/so101_mjlab/env_cfgs.py": '"""SO-ARM101 manipulation environment configurations for mjlab."""\n\nimport math\nfrom typing import Any, Literal\n\nimport mujoco\nfrom mjlab.entity import EntityCfg\nfrom mjlab.envs import ManagerBasedRlEnvCfg\nfrom mjlab.envs.mdp import dr\nfrom mjlab.envs.mdp.actions import JointPositionActionCfg\nfrom mjlab.managers import ObservationGroupCfg, ObservationTermCfg\nfrom mjlab.managers.event_manager import EventTermCfg\nfrom mjlab.managers.scene_entity_config import SceneEntityCfg\nfrom mjlab.sensor import CameraSensorCfg, ContactSensorCfg\nfrom mjlab.tasks.manipulation import mdp as manipulation_mdp\nfrom mjlab.tasks.manipulation.lift_cube_env_cfg import make_lift_cube_env_cfg\nfrom mjlab.tasks.manipulation.mdp import LiftingCommandCfg\n\nfrom so101_mjlab.so101.so101_constants import (\n  SO101_ACTION_SCALE,\n  get_so101_robot_cfg,\n)\n\n\ndef get_cube_spec(\n  cube_size: float = 0.015,\n  mass: float = 0.03,\n  rgba: tuple[float, float, float, float] = (0.85, 0.2, 0.2, 1.0),\n) -> mujoco.MjSpec:\n  """Create a 3 cm, 30 g cube sized for the SO-101 parallel jaw gripper."""\n  spec = mujoco.MjSpec()\n  body = spec.worldbody.add_body(name="cube")\n  body.add_freejoint(name="cube_joint")\n  body.add_geom(\n    name="cube_geom",\n    type=mujoco.mjtGeom.mjGEOM_BOX,\n    size=(cube_size,) * 3,\n    mass=mass,\n    rgba=rgba,\n  )\n  return spec\n\n\ndef so101_lift_cube_env_cfg(\n  play: bool = False,\n) -> ManagerBasedRlEnvCfg:\n  """Create the state-based SO-101 cube-lifting environment configuration."""\n  cfg = make_lift_cube_env_cfg()\n  cfg.sim.mujoco.ls_iterations = 50\n\n  cfg.scene.entities = {\n    "robot": get_so101_robot_cfg(),\n    "cube": EntityCfg(spec_fn=get_cube_spec),\n  }\n\n  joint_pos_action = cfg.actions["joint_pos"]\n  assert isinstance(joint_pos_action, JointPositionActionCfg)\n  joint_pos_action.scale = SO101_ACTION_SCALE\n\n  # Adapt cube spawn and 3D goal workspace to the SO-101 ~30 cm reach.\n  assert cfg.commands is not None\n  lift_cmd = cfg.commands["lift_height"]\n  assert isinstance(lift_cmd, LiftingCommandCfg)\n  lift_cmd.object_pose_range = LiftingCommandCfg.ObjectPoseRangeCfg(\n    x=(0.18, 0.26),\n    y=(-0.10, 0.10),\n    z=(0.015, 0.03),\n    yaw=(-math.pi, math.pi),\n  )\n  lift_cmd.target_position_range = LiftingCommandCfg.TargetPositionRangeCfg(\n    x=(0.18, 0.26),\n    y=(-0.10, 0.10),\n    z=(0.12, 0.20),\n  )\n\n  cfg.observations["actor"].terms["ee_to_cube"].params["asset_cfg"].site_names = (\n    "pinch_site",\n  )\n  cfg.rewards["lift"].params["asset_cfg"].site_names = ("pinch_site",)\n\n  fingertip_geoms = r"(fixed|moving)_jaw_pad"\n  cfg.events["fingertip_friction_slide"].params[\n    "asset_cfg"\n  ].geom_names = fingertip_geoms\n  cfg.events["fingertip_friction_spin"].params["asset_cfg"].geom_names = fingertip_geoms\n  cfg.events["fingertip_friction_roll"].params["asset_cfg"].geom_names = fingertip_geoms\n\n  assert cfg.scene.sensors is not None\n  for sensor in cfg.scene.sensors:\n    if sensor.name == "ee_ground_collision":\n      assert isinstance(sensor, ContactSensorCfg)\n      sensor.primary.pattern = "gripper"\n\n  cfg.viewer.body_name = "base"\n  cfg.viewer.distance = 0.75\n  cfg.viewer.azimuth = 135.0\n  cfg.viewer.elevation = -25.0\n\n  if play:\n    cfg.episode_length_s = int(1e9)\n    cfg.observations["actor"].enable_corruption = False\n    cfg.curriculum = {}\n    lift_cmd.resampling_time_range = (4.0, 4.0)\n\n  return cfg\n\n\ndef so101_lift_cube_vision_env_cfg(\n  cam_type: Literal["rgb", "depth"],\n  play: bool = False,\n) -> ManagerBasedRlEnvCfg:\n  """Create the wrist-camera (RGB or Depth) SO-101 cube-lifting configuration."""\n  cfg = so101_lift_cube_env_cfg(play=play)\n\n  camera_names = ["robot/wrist_cam"]\n  cam_kwargs = {\n    "robot/wrist_cam": {\n      "height": 32,\n      "width": 32,\n    },\n  }\n  shared_cam_kwargs = dict(\n    data_types=(cam_type,),\n    enabled_geom_groups=(0, 3),\n    use_shadows=False,\n    use_textures=True,\n  )\n\n  cam_terms = {}\n  for cam_name in camera_names:\n    cam_cfg = CameraSensorCfg(\n      name=cam_name.split("/")[-1],\n      camera_name=cam_name,\n      **cam_kwargs[cam_name],  # type: ignore[invalid-argument-type]\n      **shared_cam_kwargs,\n    )\n    cfg.scene.sensors = (cfg.scene.sensors or ()) + (cam_cfg,)\n    param_kwargs: dict[str, Any] = {"sensor_name": cam_cfg.name}\n    if cam_type == "depth":\n      param_kwargs["cutoff_distance"] = 0.4\n      func = manipulation_mdp.camera_depth\n    else:\n      func = manipulation_mdp.camera_rgb\n    cam_terms[f"{cam_name.split(\'/\')[-1]}_{cam_type}"] = ObservationTermCfg(\n      func=func, params=param_kwargs\n    )\n\n  camera_obs = ObservationGroupCfg(\n    terms=cam_terms, enable_corruption=False, concatenate_terms=True\n  )\n  cfg.observations["camera"] = camera_obs\n\n  if cam_type == "rgb":\n    cfg.events["cube_color"] = EventTermCfg(\n      func=dr.geom_rgba,\n      mode="reset",\n      params={\n        "asset_cfg": SceneEntityCfg("cube", geom_names=(".*",)),\n        "operation": "abs",\n        "distribution": "uniform",\n        "axes": [0, 1, 2],\n        "ranges": (0.0, 1.0),\n      },\n    )\n\n  actor_obs = cfg.observations["actor"]\n  actor_obs.terms.pop("ee_to_cube")\n  actor_obs.terms.pop("cube_to_goal")\n\n  actor_obs.terms["goal_position"] = ObservationTermCfg(\n    func=manipulation_mdp.target_position,\n    params={\n      "command_name": "lift_height",\n      "asset_cfg": SceneEntityCfg("robot", site_names=("pinch_site",)),\n    },\n  )\n\n  return cfg\n',
  "src/so101_mjlab/rl_cfg.py": '"""RSL-RL PPO runner configurations for SO-101 manipulation tasks."""\n\nfrom mjlab.rl import (\n  RslRlModelCfg,\n  RslRlOnPolicyRunnerCfg,\n  RslRlPpoAlgorithmCfg,\n)\n\n_VISION_CNN_CFG = {\n  "output_channels": [16, 32],\n  "kernel_size": [5, 3],\n  "stride": [2, 2],\n  "padding": "zeros",\n  "activation": "elu",\n  "max_pool": False,\n  "global_pool": "none",\n  "spatial_softmax": True,\n  "spatial_softmax_temperature": 1.0,\n}\n_VISION_MODEL_CLS = "mjlab.rl.spatial_softmax:SpatialSoftmaxCNNModel"\n\n\ndef so101_lift_cube_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:\n  return RslRlOnPolicyRunnerCfg(\n    actor=RslRlModelCfg(\n      hidden_dims=(512, 256, 128),\n      activation="elu",\n      obs_normalization=True,\n      distribution_cfg={\n        "class_name": "GaussianDistribution",\n        "init_std": 1.0,\n        "std_type": "scalar",\n      },\n    ),\n    critic=RslRlModelCfg(\n      hidden_dims=(512, 256, 128),\n      activation="elu",\n      obs_normalization=True,\n    ),\n    algorithm=RslRlPpoAlgorithmCfg(\n      value_loss_coef=1.0,\n      use_clipped_value_loss=True,\n      clip_param=0.2,\n      entropy_coef=0.005,\n      num_learning_epochs=5,\n      num_mini_batches=4,\n      learning_rate=1.0e-3,\n      schedule="adaptive",\n      gamma=0.99,\n      lam=0.95,\n      desired_kl=0.01,\n      max_grad_norm=1.0,\n    ),\n    experiment_name="so101_lift_cube",\n    save_interval=100,\n    num_steps_per_env=24,\n    max_iterations=2_500,\n  )\n\n\ndef so101_lift_cube_vision_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:\n  return RslRlOnPolicyRunnerCfg(\n    actor=RslRlModelCfg(\n      hidden_dims=(256, 256, 128),\n      activation="elu",\n      obs_normalization=True,\n      cnn_cfg=_VISION_CNN_CFG,\n      class_name=_VISION_MODEL_CLS,\n      distribution_cfg={\n        "class_name": "GaussianDistribution",\n        "init_std": 1.0,\n        "std_type": "scalar",\n      },\n    ),\n    critic=RslRlModelCfg(\n      hidden_dims=(256, 256, 128),\n      activation="elu",\n      obs_normalization=True,\n      cnn_cfg=_VISION_CNN_CFG,\n      class_name=_VISION_MODEL_CLS,\n    ),\n    algorithm=RslRlPpoAlgorithmCfg(\n      value_loss_coef=1.0,\n      use_clipped_value_loss=True,\n      clip_param=0.2,\n      entropy_coef=0.005,\n      num_learning_epochs=5,\n      num_mini_batches=4,\n      learning_rate=1.0e-3,\n      schedule="adaptive",\n      gamma=0.99,\n      lam=0.95,\n      desired_kl=0.01,\n      max_grad_norm=1.0,\n    ),\n    experiment_name="so101_lift_cube_vision",\n    save_interval=100,\n    num_steps_per_env=24,\n    max_iterations=3_000,\n    obs_groups={\n      "actor": ("actor", "camera"),\n      "critic": ("critic", "camera"),\n    },\n  )\n',
  "src/so101_mjlab/deploy_real.py": '"""Closed-loop 50 Hz ONNX policy deployment for the physical SO-ARM101 follower arm."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport glob\nimport time\nfrom pathlib import Path\n\nimport mujoco\nimport numpy as np\n\nfrom so101_mjlab.so101.so101_constants import (\n  HOME_KEYFRAME,\n  SO101_ACTION_SCALE,\n  SO101_XML,\n)\n\nDEFAULT_PORT = "/dev/tty.usbmodem5B790143431"\nKNOWN_LEADER_SUBSTR = "5B8E116931"\nBAUD = 1_000_000\nZERO_OFFSETS = np.array(\n  [2027.5, 1851.0, 2035.0, 1930.0, 1997.0, 2048.0], dtype=np.float64\n)\nRAD_PER_TICK = (2.0 * np.pi) / 4096.0\n\nJOINT_NAMES = (\n  "shoulder_pan",\n  "shoulder_lift",\n  "elbow_flex",\n  "wrist_flex",\n  "wrist_roll",\n  "gripper",\n)\n\nDEFAULT_JOINT_POS = np.array(\n  [HOME_KEYFRAME.joint_pos[name] for name in JOINT_NAMES], dtype=np.float32\n)\nACTION_SCALE = np.array(\n  [SO101_ACTION_SCALE[name] for name in JOINT_NAMES], dtype=np.float32\n)\n\n\ndef _checksum(packet_body: list[int]) -> int:\n  return (~sum(packet_body)) & 0xFF\n\n\ndef read_u16(ser, motor_id: int, reg: int) -> int:\n  body = [motor_id, 4, 2, reg, 2]\n  ser.reset_input_buffer()\n  ser.write(bytes([0xFF, 0xFF, *body, _checksum(body)]))\n  resp = ser.read(8)\n  if len(resp) == 8 and resp[0] == 0xFF and resp[1] == 0xFF:\n    return int(resp[5] | (resp[6] << 8))\n  return -1\n\n\ndef write_u8(ser, motor_id: int, reg: int, val: int) -> None:\n  body = [motor_id, 4, 3, reg, val & 0xFF]\n  ser.write(bytes([0xFF, 0xFF, *body, _checksum(body)]))\n  ser.read(6)\n\n\ndef write_u16(ser, motor_id: int, reg: int, val: int) -> None:\n  lo = val & 0xFF\n  hi = (val >> 8) & 0xFF\n  body = [motor_id, 5, 3, reg, lo, hi]\n  ser.write(bytes([0xFF, 0xFF, *body, _checksum(body)]))\n  ser.read(6)\n\n\ndef open_follower_serial(port: str | None = None):\n  import serial\n\n  if port is not None:\n    return serial.Serial(port, BAUD, timeout=0.02)\n\n  all_usb = sorted(glob.glob("/dev/tty.usbmodem*"))\n  candidates: list[str] = []\n  if Path(DEFAULT_PORT).exists():\n    candidates.append(DEFAULT_PORT)\n  for p in all_usb:\n    if p not in candidates and KNOWN_LEADER_SUBSTR not in p:\n      candidates.append(p)\n\n  if not candidates:\n    raise RuntimeError(\n      "No follower /dev/tty.usbmodem* port detected. Check USB-C connection."\n    )\n\n  for candidate in candidates:\n    ser = serial.Serial(candidate, BAUD, timeout=0.02)\n    if read_u16(ser, 1, 56) >= 0:\n      return ser\n    ser.close()\n\n  raise RuntimeError(\n    f"Follower serial port found ({candidates}), but STS3215 servos did not respond. "\n    "Verify that 12V power is switched on."\n  )\n\n\ndef ticks_to_rad(ticks: list[int]) -> np.ndarray:\n  return ((np.asarray(ticks, dtype=np.float64) - ZERO_OFFSETS) * RAD_PER_TICK).astype(\n    np.float32\n  )\n\n\ndef rad_to_ticks(q_rad: np.ndarray) -> list[int]:\n  raw = np.round(ZERO_OFFSETS + (q_rad.astype(np.float64) / RAD_PER_TICK)).astype(int)\n  return [int(np.clip(v, 200, 3900)) for v in raw]\n\n\ndef read_joint_positions(ser) -> np.ndarray:\n  ticks: list[int] = []\n  for m in range(1, 7):\n    val = read_u16(ser, m, 56)\n    if val < 0:\n      val = read_u16(ser, m, 56)\n    if val < 0:\n      raise RuntimeError(f"Lost communication with STS3215 servo ID {m}.")\n    ticks.append(val)\n  return ticks_to_rad(ticks)\n\n\ndef command_joint_positions(ser, q_target_rad: np.ndarray) -> None:\n  target_ticks = rad_to_ticks(q_target_rad)\n  for m in range(1, 7):\n    write_u16(ser, m, 42, target_ticks[m - 1])\n\n\ndef interpolate_to_pose(\n  ser, q_target: np.ndarray, duration_s: float = 2.0, rate_hz: float = 50.0\n) -> None:\n  q_start = read_joint_positions(ser)\n  steps = max(1, int(duration_s * rate_hz))\n  dt = 1.0 / rate_hz\n  for m in range(1, 7):\n    write_u8(ser, m, 40, 1)\n  for step in range(1, steps + 1):\n    alpha = 0.5 * (1.0 - np.cos(np.pi * (step / steps)))\n    q_cmd = (1.0 - alpha) * q_start + alpha * q_target\n    command_joint_positions(ser, q_cmd)\n    time.sleep(dt)\n\n\ndef relax_servos(ser) -> None:\n  for m in range(1, 7):\n    write_u8(ser, m, 40, 0)\n\n\ndef build_state_observation(\n  mj_model: mujoco.MjModel,\n  mj_data: mujoco.MjData,\n  site_id: int,\n  q_rad: np.ndarray,\n  dq_rad: np.ndarray,\n  cube_pos_b: np.ndarray,\n  goal_pos_b: np.ndarray,\n  last_action: np.ndarray,\n) -> np.ndarray:\n  """Construct the 24-D actor observation vector matching Mjlab-Lift-Cube-SO101."""\n  mj_data.qpos[:6] = q_rad\n  mujoco.mj_kinematics(mj_model, mj_data)\n  ee_pos_b = mj_data.site_xpos[site_id].astype(np.float32)\n\n  joint_pos_rel = q_rad - DEFAULT_JOINT_POS\n  joint_vel_rel = dq_rad\n  ee_to_cube = cube_pos_b - ee_pos_b\n  cube_to_goal = goal_pos_b - cube_pos_b\n\n  obs = np.concatenate(\n    [joint_pos_rel, joint_vel_rel, ee_to_cube, cube_to_goal, last_action],\n    axis=0,\n  ).astype(np.float32)\n  return obs.reshape(1, -1)\n\n\ndef main() -> None:\n  parser = argparse.ArgumentParser(\n    description="Deploy an mjlab ONNX manipulation policy to the physical SO-101 arm."\n  )\n  parser.add_argument(\n    "--onnx",\n    type=Path,\n    default=None,\n    help="Path to exported ONNX policy file (if omitted, runs zero-action holding test).",\n  )\n  parser.add_argument(\n    "--port",\n    type=str,\n    default=None,\n    help="Serial port override (auto-detects follower /dev/tty.usbmodem* by default).",\n  )\n  parser.add_argument(\n    "--cube-xyz",\n    type=float,\n    nargs=3,\n    default=[0.22, 0.0, 0.015],\n    help="Initial cube position [x, y, z] in base frame (meters).",\n  )\n  parser.add_argument(\n    "--goal-xyz",\n    type=float,\n    nargs=3,\n    default=[0.22, 0.0, 0.16],\n    help="Target lift goal [x, y, z] in base frame (meters).",\n  )\n  parser.add_argument(\n    "--steps",\n    type=int,\n    default=250,\n    help="Number of 50 Hz control steps to execute (default: 250 = 5.0 seconds).",\n  )\n  parser.add_argument(\n    "--max-delta-rad",\n    type=float,\n    default=0.15,\n    help="Per-step joint command safety clamp relative to current pose (radians).",\n  )\n  parser.add_argument(\n    "--dry-run",\n    action="store_true",\n    help="Run kinematic & ONNX inference loop in simulation without opening USB serial.",\n  )\n  args = parser.parse_args()\n\n  mj_model = mujoco.MjModel.from_xml_path(str(SO101_XML))\n  mj_data = mujoco.MjData(mj_model)\n  site_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_SITE, "pinch_site")\n\n  ort_session = None\n  input_name = None\n  if args.onnx is not None:\n    import onnxruntime as ort\n\n    ort_session = ort.InferenceSession(\n      str(args.onnx), providers=["CPUExecutionProvider"]\n    )\n    input_name = ort_session.get_inputs()[0].name\n\n  cube_pos_b = np.asarray(args.cube_xyz, dtype=np.float32)\n  goal_pos_b = np.asarray(args.goal_xyz, dtype=np.float32)\n  last_action = np.zeros(6, dtype=np.float32)\n  q_prev = DEFAULT_JOINT_POS.copy()\n  dt = 0.02  # 50 Hz control rate\n\n  if args.dry_run:\n    print("Running in --dry-run mode (no hardware serial required).")\n    q_cur = DEFAULT_JOINT_POS.copy()\n    for step in range(args.steps):\n      dq_cur = (q_cur - q_prev) / dt\n      q_prev = q_cur.copy()\n      obs = build_state_observation(\n        mj_model,\n        mj_data,\n        site_id,\n        q_cur,\n        dq_cur,\n        cube_pos_b,\n        goal_pos_b,\n        last_action,\n      )\n      if ort_session is not None and input_name is not None:\n        raw_action = np.asarray(ort_session.run(None, {input_name: obs})[0]).reshape(\n          -1\n        )[:6]\n      else:\n        raw_action = np.zeros(6, dtype=np.float32)\n      raw_action = np.clip(raw_action, -1.0, 1.0)\n      q_target = DEFAULT_JOINT_POS + ACTION_SCALE * raw_action\n      q_cur = q_cur + 0.25 * (q_target - q_cur)\n      last_action = raw_action\n      if step % 50 == 0 or step == args.steps - 1:\n        ee_pos = mj_data.site_xpos[site_id]\n        print(\n          f"[step {step:03d}] pinch_site={np.round(ee_pos, 4)} "\n          f"q_target={np.round(q_target, 3)}"\n        )\n    return\n\n  ser = open_follower_serial(args.port)\n  try:\n    print("Moving SO-101 smoothly to HOME_KEYFRAME...")\n    interpolate_to_pose(ser, DEFAULT_JOINT_POS, duration_s=2.0, rate_hz=50.0)\n    q_prev = read_joint_positions(ser)\n\n    print(f"Running 50 Hz closed-loop policy for {args.steps} steps...")\n    for step in range(args.steps):\n      t0 = time.perf_counter()\n      q_cur = read_joint_positions(ser)\n      dq_cur = (q_cur - q_prev) / dt\n      q_prev = q_cur.copy()\n\n      obs = build_state_observation(\n        mj_model,\n        mj_data,\n        site_id,\n        q_cur,\n        dq_cur,\n        cube_pos_b,\n        goal_pos_b,\n        last_action,\n      )\n      if ort_session is not None and input_name is not None:\n        raw_action = np.asarray(ort_session.run(None, {input_name: obs})[0]).reshape(\n          -1\n        )[:6]\n      else:\n        raw_action = np.zeros(6, dtype=np.float32)\n\n      raw_action = np.clip(raw_action, -1.0, 1.0)\n      q_des = DEFAULT_JOINT_POS + ACTION_SCALE * raw_action\n      q_cmd = np.clip(q_des, q_cur - args.max_delta_rad, q_cur + args.max_delta_rad)\n      command_joint_positions(ser, q_cmd)\n      last_action = raw_action\n      if step % 50 == 0 or step == args.steps - 1:\n        ee_pos = mj_data.site_xpos[site_id]\n        print(\n          f"[step {step:03d}] pinch_site={np.round(ee_pos, 4)} "\n          f"q_cmd={np.round(q_cmd, 3)}"\n        )\n\n      elapsed = time.perf_counter() - t0\n      if elapsed < dt:\n        time.sleep(dt - elapsed)\n\n    print("Returning SO-101 smoothly to HOME_KEYFRAME...")\n    interpolate_to_pose(ser, DEFAULT_JOINT_POS, duration_s=1.5, rate_hz=50.0)\n  finally:\n    relax_servos(ser)\n    ser.close()\n\n\nif __name__ == "__main__":\n  main()\n',
  "src/so101_mjlab/so101/__init__.py": '"""SO-101 robot asset definition for mjlab."""\n\nfrom .so101_constants import (\n  SO101_ACTION_SCALE,\n  SO101_ARTICULATION,\n  SO101_XML,\n  get_so101_robot_cfg,\n  get_spec,\n)\n\n__all__ = [\n  "SO101_ACTION_SCALE",\n  "SO101_ARTICULATION",\n  "SO101_XML",\n  "get_so101_robot_cfg",\n  "get_spec",\n]\n',
  "src/so101_mjlab/so101/so101_constants.py": '"""SO-ARM101 constants and robot configuration for mjlab."""\n\nimport math\nfrom pathlib import Path\n\nimport mujoco\nfrom mjlab.actuator import BuiltinPositionActuatorCfg\nfrom mjlab.entity import EntityArticulationInfoCfg, EntityCfg\nfrom mjlab.utils.spec_config import CollisionCfg\n\n##\n# MJCF and assets.\n##\n\n_HERE = Path(__file__).parent\n\nSO101_XML: Path = _HERE / "xmls" / "so101.xml"\nassert SO101_XML.exists()\n\n\ndef get_spec() -> mujoco.MjSpec:\n  return mujoco.MjSpec.from_file(str(SO101_XML))\n\n\n##\n# Actuator config (Feetech STS3215 12V 30 kg.cm servo).\n##\n\nEFFORT_LIMIT = 3.35\nARMATURE = 0.028\n\nNATURAL_FREQ = 4.0 * 2.0 * math.pi  # 4 Hz closed-loop bandwidth\nDAMPING_RATIO = 1.5\n\nSTIFFNESS = ARMATURE * NATURAL_FREQ**2\nDAMPING = 2.0 * DAMPING_RATIO * ARMATURE * NATURAL_FREQ\n\nSO101_ARM_ACTUATOR_CFG = BuiltinPositionActuatorCfg(\n  target_names_expr=(\n    "shoulder_pan",\n    "shoulder_lift",\n    "elbow_flex",\n    "wrist_flex",\n    "wrist_roll",\n  ),\n  stiffness=STIFFNESS,\n  damping=DAMPING,\n  effort_limit=EFFORT_LIMIT,\n  armature=ARMATURE,\n)\n\nSO101_GRIPPER_ACTUATOR_CFG = BuiltinPositionActuatorCfg(\n  target_names_expr=("gripper",),\n  stiffness=STIFFNESS,\n  damping=DAMPING,\n  effort_limit=EFFORT_LIMIT * 0.6,\n  armature=ARMATURE,\n)\n\n##\n# Keyframe config.\n##\n\nHOME_KEYFRAME = EntityCfg.InitialStateCfg(\n  pos=(0.0, 0.0, 0.005),\n  joint_pos={\n    "shoulder_pan": 0.0,\n    "shoulder_lift": -0.35,\n    "elbow_flex": 0.70,\n    "wrist_flex": 0.95,\n    "wrist_roll": -1.5708,\n    "gripper": 0.45,\n  },\n  joint_vel={".*": 0.0},\n)\n\n##\n# Collision config.\n##\n\nFULL_COLLISION = CollisionCfg(\n  geom_names_expr=(".*_collision", ".*_pad", "gripper_palm_col"),\n  contype=1,\n  conaffinity=1,\n  condim={\n    ".*_pad": 6,\n    ".*": 3,\n  },\n  friction={\n    ".*_pad": (1.5, 5e-3, 5e-4),\n    ".*": (0.8,),\n  },\n  solref={\n    ".*_pad": (0.008, 1.0),\n  },\n  priority={\n    ".*_pad": 1,\n    ".*": 0,\n  },\n)\n\n##\n# Final config.\n##\n\nSO101_ARTICULATION = EntityArticulationInfoCfg(\n  actuators=(SO101_ARM_ACTUATOR_CFG, SO101_GRIPPER_ACTUATOR_CFG),\n  soft_joint_pos_limit_factor=0.9,\n)\n\n\ndef get_so101_robot_cfg() -> EntityCfg:\n  """Return a fresh SO-101 EntityCfg instance."""\n  return EntityCfg(\n    init_state=HOME_KEYFRAME,\n    collisions=(FULL_COLLISION,),\n    spec_fn=get_spec,\n    articulation=SO101_ARTICULATION,\n  )\n\n\nSO101_ACTION_SCALE: dict[str, float] = {\n  "shoulder_pan": 0.5,\n  "shoulder_lift": 0.5,\n  "elbow_flex": 0.5,\n  "wrist_flex": 0.5,\n  "wrist_roll": 0.5,\n  "gripper": 0.5,\n}\n\n\nif __name__ == "__main__":\n  import mujoco.viewer as viewer\n  from mjlab.entity.entity import Entity\n\n  robot = Entity(get_so101_robot_cfg())\n  viewer.launch(robot.spec.compile())\n',
  "src/so101_mjlab/so101/xmls/so101.xml": '<mujoco model="so101">\n  <compiler angle="radian" meshdir="assets" autolimits="true"/>\n\n  <default>\n    <default class="so101">\n      <site group="5" size="0.005" rgba="1 0.2 0.2 0.8"/>\n      <joint axis="0 0 1" damping="0.60" frictionloss="0.052" armature="0.028"/>\n      <default class="visual">\n        <geom type="mesh" contype="0" conaffinity="0" density="0" group="2"/>\n      </default>\n      <default class="collision">\n        <geom type="capsule" contype="1" conaffinity="1" group="3" friction="0.8 0.02 0.01" rgba="0.38 0.49 0.78 0.5"/>\n      </default>\n      <default class="jaw_pad">\n        <geom type="box" contype="1" conaffinity="1" group="3" friction="1.8 0.05 0.02" solimp="0.95 0.99 0.001" solref="0.008 1" rgba="0.15 0.15 0.15 1"/>\n      </default>\n    </default>\n  </default>\n\n  <asset>\n    <mesh file="waveshare_mounting_plate_so101_v2.stl"/>\n    <mesh file="sts3215_03a_v1.stl"/>\n    <mesh file="motor_holder_so101_base_v1.stl"/>\n    <mesh file="wrist_roll_follower_so101_v1.stl"/>\n    <mesh file="moving_jaw_so101_v1.stl"/>\n    <mesh file="base_motor_holder_so101_v1.stl"/>\n    <mesh file="upper_arm_so101_v1.stl"/>\n    <mesh file="wrist_roll_pitch_so101_v2.stl"/>\n    <mesh file="under_arm_so101_v1.stl"/>\n    <mesh file="rotation_pitch_so101_v1.stl"/>\n    <mesh file="motor_holder_so101_wrist_v1.stl"/>\n    <mesh file="sts3215_03a_no_horn_v1.stl"/>\n    <mesh file="base_so101_v2.stl"/>\n    <mesh file="wrist_camera_mount_so101_v1.stl" scale="0.001 0.001 0.001"/>\n    <mesh file="wrist_camera_so101_v1.stl"/>\n\n    <material name="so101_body_mat" rgba="0.96 0.78 0.14 1" specular="0.3" shininess="0.4"/>\n    <material name="sts3215_servo_mat" rgba="0.12 0.12 0.14 1" specular="0.2" shininess="0.3"/>\n  </asset>\n\n  <worldbody>\n    <light name="spotlight" mode="targetbodycom" target="gripper" pos="0 0 1"/>\n    <body name="base" pos="0 0 0" quat="1 0 0 0" childclass="so101">\n      <inertial pos="0.0137179 -5.19711e-05 0.0334843" mass="0.147" fullinertia="0.000114686 0.000136117 0.000130364 -4.59787e-07 4.97151e-06 9.75275e-08"/>\n      <geom class="visual" pos="-0.00636471 -9.94414e-05 -0.0024" quat="0.5 0.5 0.5 0.5" mesh="base_motor_holder_so101_v1" material="so101_body_mat"/>\n      <geom class="visual" pos="-0.00636471 -8.97657e-09 -0.0024" quat="0.5 0.5 0.5 0.5" mesh="base_so101_v2" material="so101_body_mat"/>\n      <geom class="visual" pos="0.0263353 -8.97657e-09 0.0437" quat="1 0 0 0" mesh="sts3215_03a_v1" material="sts3215_servo_mat"/>\n      <geom class="visual" pos="-0.0309827 -0.000199441 0.0474" quat="0.5 0.5 0.5 0.5" mesh="waveshare_mounting_plate_so101_v2" material="so101_body_mat"/>\n      <geom name="base_collision" class="collision" size="0.035 0.025" pos="0.01 0.0 0.03"/>\n      <site name="baseframe" pos="0 0 0" quat="1 0 0 0"/>\n\n      <body name="shoulder" pos="0.0388353 0 0.0624" quat="0 0 -1 0">\n        <joint name="shoulder_pan" type="hinge" range="-1.91986 1.91986"/>\n        <inertial pos="-0.0307604 -1.66727e-05 -0.0252713" mass="0.100006" fullinertia="8.3759e-05 8.10403e-05 2.39783e-05 7.55525e-08 -1.16342e-06 1.54663e-07"/>\n        <geom class="visual" pos="-0.0303992 0.000422241 -0.0417" quat="0.5 0.5 0.5 -0.5" mesh="sts3215_03a_v1" material="sts3215_servo_mat"/>\n        <geom class="visual" pos="-0.0675992 -0.000177759 0.0158499" quat="0.5 0.5 -0.5 0.5" mesh="motor_holder_so101_base_v1" material="so101_body_mat"/>\n        <geom class="visual" pos="0.0122008 2.22413e-05 0.0464" quat="0.707107 -0.707107 0 0" mesh="rotation_pitch_so101_v1" material="so101_body_mat"/>\n\n        <body name="upper_arm" pos="-0.0303992 -0.0182778 -0.0542" quat="0.5 -0.5 -0.5 -0.5">\n          <joint name="shoulder_lift" type="hinge" range="-1.74533 1.74533"/>\n          <inertial pos="-0.0898471 -0.00838224 0.0184089" mass="0.103" fullinertia="4.08002e-05 0.000147318 0.000142487 -1.97819e-05 -4.03016e-08 8.97326e-09"/>\n          <geom class="visual" pos="-0.11257 -0.0155 0.0187" quat="0 -0.707107 0.707107 0" mesh="sts3215_03a_v1" material="sts3215_servo_mat"/>\n          <geom class="visual" pos="-0.065085 0.012 0.0182" quat="0 1 0 0" mesh="upper_arm_so101_v1" material="so101_body_mat"/>\n          <geom name="upper_arm_collision" class="collision" fromto="-0.01 0.0 0.018 -0.105 -0.022 0.018" size="0.022"/>\n\n          <body name="lower_arm" pos="-0.11257 -0.028 0" quat="0.707107 0 0 0.707107">\n            <joint name="elbow_flex" type="hinge" range="-1.69 1.69"/>\n            <inertial pos="-0.0980701 0.00324376 0.0182831" mass="0.104" fullinertia="2.87438e-05 0.000159844 0.00014529 7.41152e-06 1.26409e-06 -4.90188e-08"/>\n            <geom class="visual" pos="-0.0648499 -0.032 0.0182" quat="0 1 0 0" mesh="under_arm_so101_v1" material="so101_body_mat"/>\n            <geom class="visual" pos="-0.0648499 -0.032 0.018" quat="0 -1 0 0" mesh="motor_holder_so101_wrist_v1" material="so101_body_mat"/>\n            <geom class="visual" pos="-0.1224 0.0052 0.0187" quat="0 0 1 0" mesh="sts3215_03a_v1" material="sts3215_servo_mat"/>\n            <geom name="lower_arm_collision" class="collision" fromto="-0.01 -0.01 0.018 -0.125 0.005 0.018" size="0.020"/>\n\n            <body name="wrist" pos="-0.1349 0.0052 0" quat="0.707107 0 0 -0.707107">\n              <joint name="wrist_flex" type="hinge" range="-1.65806 1.65806"/>\n              <inertial pos="-0.000103312 -0.0386143 0.0281156" mass="0.079" fullinertia="3.68263e-05 2.5391e-05 2.1e-05 1.7893e-08 -5.28128e-08 3.6412e-06"/>\n              <geom class="visual" pos="0 -0.0424 0.0306" quat="0.5 0.5 0.5 -0.5" mesh="sts3215_03a_no_horn_v1" material="sts3215_servo_mat"/>\n              <geom class="visual" pos="0 -0.028 0.0181" quat="0.5 -0.5 -0.5 -0.5" mesh="wrist_roll_pitch_so101_v2" material="so101_body_mat"/>\n              <geom name="wrist_collision" class="collision" fromto="0.0 -0.015 0.022 0.0 -0.045 0.022" size="0.018"/>\n\n              <body name="gripper" pos="0 -0.0611 0.0181" quat="0.0172091 -0.0172091 0.706897 0.706897">\n                <joint name="wrist_roll" type="hinge" range="-2.74385 2.84121"/>\n                <inertial pos="0.000213627 0.000245138 -0.025187" mass="0.087" fullinertia="2.75087e-05 4.33657e-05 3.45059e-05 -3.35241e-07 -5.7352e-06 -5.17847e-08"/>\n                <geom class="visual" pos="0.0077 0.0001 -0.0234" quat="0.707107 -0.707107 0 0" mesh="sts3215_03a_v1" material="sts3215_servo_mat"/>\n                <geom class="visual" pos="0 -0.000218214 0.000949706" quat="0 1 0 0" mesh="wrist_roll_follower_so101_v1" material="so101_body_mat"/>\n\n                <geom name="fixed_jaw_pad" class="jaw_pad" pos="-0.013 0.0 -0.082" size="0.004 0.008 0.020"/>\n                <geom name="gripper_palm_col" type="box" contype="1" conaffinity="1" group="3" pos="0.0 0.0 -0.032" size="0.020 0.018 0.016" rgba="0.38 0.49 0.78 0.5"/>\n\n                <site name="gripperframe" pos="-0.0079 -0.000218121 -0.0981274" quat="0.707107 0 0.707107 0"/>\n                <site name="pinch_site" pos="0.002 -0.004 -0.082" quat="1 0 0 0"/>\n                <site name="grasp_site" pos="0.002 -0.004 -0.082" quat="1 0 0 0"/>\n\n                <body name="wrist_camera_mount" pos="-0.0150 0.0240 -0.0315" quat="0.499798 -0.500200 0.499802 -0.500200">\n                  <inertial pos="0 0 0" mass="0.02" diaginertia="1e-5 1e-5 1e-5"/>\n                  <geom class="visual" mesh="wrist_camera_mount_so101_v1" material="so101_body_mat"/>\n                  <body name="wrist_camera" pos="-0.04941 -0.03813 0.0175" quat="0.379787 -0.596458 -0.379788 -0.596456">\n                    <inertial pos="0 0 0" mass="0.015" diaginertia="1e-5 1e-5 1e-5"/>\n                    <geom class="visual" mesh="wrist_camera_so101_v1" material="sts3215_servo_mat"/>\n                    <camera name="wrist_cam" pos="0.002 0.005 0.024" quat="0.0 0.9925 0.1219 0.0" fovy="75"/>\n                  </body>\n                </body>\n\n                <body name="moving_jaw_so101_v1" pos="0.0202 0.0188 -0.0234" quat="0.707107 0.707107 0 0">\n                  <joint name="gripper" type="hinge" range="-0.174533 1.74533"/>\n                  <inertial pos="-0.00157495 -0.0300244 0.0192755" mass="0.012" fullinertia="6.61427e-06 1.89032e-06 5.28738e-06 -3.19807e-07 -5.90717e-09 -1.09945e-07"/>\n                  <geom class="visual" pos="0 0 0.0189" quat="1 0 0 0" mesh="moving_jaw_so101_v1" material="so101_body_mat"/>\n                  <geom name="moving_jaw_pad" class="jaw_pad" pos="-0.006 -0.060 0.019" size="0.004 0.020 0.008"/>\n                </body>\n              </body>\n            </body>\n          </body>\n        </body>\n      </body>\n    </body>\n  </worldbody>\n</mujoco>\n',
}
for rel_path, content in TEXT_FILES.items():
  p = REPO_DIR / rel_path
  p.parent.mkdir(parents=True, exist_ok=True)
  p.write_text(content)

UPSTREAM_STLS = [
  "waveshare_mounting_plate_so101_v2.stl",
  "sts3215_03a_v1.stl",
  "motor_holder_so101_base_v1.stl",
  "wrist_roll_follower_so101_v1.stl",
  "moving_jaw_so101_v1.stl",
  "base_motor_holder_so101_v1.stl",
  "upper_arm_so101_v1.stl",
  "wrist_roll_pitch_so101_v2.stl",
  "under_arm_so101_v1.stl",
  "rotation_pitch_so101_v1.stl",
  "motor_holder_so101_wrist_v1.stl",
  "sts3215_03a_no_horn_v1.stl",
  "base_so101_v2.stl",
]
BASE_URL = "https://raw.githubusercontent.com/TheRobotStudio/SO-ARM100/main/Simulation/SO101/assets"
for stl in UPSTREAM_STLS:
  dest = ASSETS_DIR / stl
  if not dest.exists():
    urllib.request.urlretrieve(f"{BASE_URL}/{stl}", dest)

EXTRA_STLS = {
  "wrist_camera_mount_so101_v1.stl": "{Wp48S^xk9=GL@E0stWa761SMbT8$j;-@=cXk7p)l?9quuu<~{{6qB7;m9PE*%vk;`T1zO6RqU*;goS(&qm{CnXw?$Rob8c30Vner^@Dld*PfupT)0eFIv3{o2Ga(=MO`X?_1}8{W|R8?nImf%BfKTFL}SQ=(d1M6E?=nP^!c8_wtWv=xtP#97qh|mj%}+J(D}6)U0j7^<vekfKH9|s+*jNfgbX~cu9IYUgl3ltO8{pArl(p)rAZweqDX#J_u#kHmw#-9beh?In+80bg-k%)t3KID*vhgC6D49k@efG0)9*Ji^}~^@|TkKU}SzUXAHZ$U~@W3i2X>lbd&VNDp&wy)%A_^9CMp(jYNkwLq39!GO7ZU^l6=-mkO~$3iSD1zg|GoWnP9@_@*~<qJevOAw0|4RO79wXb?(&<NE+8Rw}LSo{2YNcn5<)?j*z!Fd<$QScvGoJ#hAR4>LVkx(zBmadX3A%-MM(XNVD&akrNghc=F08yJ-@`9}Eb0qKotLKe!y9IlTPBJ9~|z0e?A-WR2W8+aCLOxLee{NlJkS6?)Mhx_c6`<|iOk?aJHT9Q&QhUdPfR`F(5AvXX%7K=Sp8q%sh>vEM6MhG5Q4E2h34)L0dT0Ji9p6!?RzFJ*`<=0}d+bSEn+yIDMt?x8BT1Xn{U~zeWHIH|rsP4V>i{6E9O}d#XK%MGhuyeNbt#%$So-nGRm3Ig)Sug{J=*aZJy$dY|SxRINu4ohI9jgT_Xy;TDD~)gV?8=DaQ%Yl_#H4IDzbyO};pN#9Bzi~jDy?RC83#j*pSCYx=-G2faeXzoVGMLqM<(aEX&ZSsjV4H3l&2!SNVLG>v&X_bH3Ov&rYf>kSyL&&qDF5+d1aGCJMp`H#8OH20kW6cMET%Q+{Hv7Z04H@K}%Mp>5yW|T_e#lo#VX4wVlEFazx0i{aY>cpAZ@BHW=*CPPN|9ITaq4*as8?MYghBBy=9+CCZz}FY_ac2$p+5g{Wgud^HX@YfWd{Z`*!0fxLwK=8NnwFhXKtbvdAUr-=Hj!aFyGTTyupLFm!TUGih%gGVfcM@>8U()$FwvsI{|Sjr>smK(jIt(J)6q9K9}Fr!4yOPmW`&qI5bJgIhL55eCm|5fcOR`h{(=b$d^`}sW7AqBG@-LxUNY|X9ii2>wvm`^l{8E*p4BfzE=@@<vFDkMf8zn3nIur^<zinStwbR>T!9!p7IDa_>4*-7S~GvD&e5&RnBUc+CqO*a~IZqEPK+f+L$q!reiYv2Z4Ty6<`iV3Vjl8CwTKW^OeL%Y#}eYoeeUN~0^T6x846DN5Ll}*-pl_KQO?%-)`lxR9_$~XPZCt@8JhSGI{opL(Ah^kHZn$Hj{1G;Zvh}9trGb9Wu-2%8`4fx%3K-|AED8LlqtSEUtSlDhT1Y)KapT=C+UI0z&cy`Y9$<9Zf7aLza@14WEjG@wxCTg+O5y_C|Hs56(c6f&Z%^D;2471k0>s>9kNI-8<VSChOQd1{aU<sl1lp7>-U&EM0p{WVA5I*m*gN6HasLSO^QlqGTEOL!QP|r0_?{}NLezng7Oz>h0i}!_P8813Bu;G#-&QcYTp=)WjK=XplO%N~=;(Se);ZIO}g<l(UFe4XD=;>2i8qCE>O%B3gllg5h;Xm~;!HRQQp2c)juXmtSfpr@nO|&Yg6)g0e!jC_z{zRD1G0$U+Aie;~N7^e!s)TIUwZ6H()gACz^#<w<XEZHp5fiN{*^?%GFEpPth&!IB#%0Yb)*JfyrYhO1!y8LM#n9j+JIejOlpLqD!Ycc8lr}`Gz#Lwcj3RrTb16Q<Flpe_gfeh{Nqo(}<4vO$9JgK@A#u&1c!B_>dz=9TzjPZWvxp1Z$!wNmJ3Ew^7!R0;OIyxaI12~E$ODgUpV*UUXMp`meaSqI#i3y)4+<tYn=$FxP={p{Wuf26F}S@Wo?bW;0OaXXlXlw}jD%i}EF1YtV54EAQ=GcWGewX$AxM<zLNio8d|a^X!&38laH(C{2wF)OQFZIoeU=&(K&4OFG%j!C#JRSKr!SU~uo#_0k><6{xB<yF9B9!F<p?ifMj>PRh(pr<uMMfTx)>(kOE(qaA9Gxf<6=8*;o?O{u}8@MU7s`zsG@u{%R-YII{-&0Z}<Swt2$-hUXu5d?=Yu@0od$;ij0YqD7W3&4D~~E^fGbJs#7Ix2$TI<>yZoWz$M2_51OzeFLe&Sr15<o0M5~ux31BeOy1t!+5i6LDWbEMSBqw^EUQ*$XM`)EO(%^tLrlTXdr7{-bUIr2IL1pbuI*7GV(B&s1}G$7U_F+#$+*0-e5Nnn9eG0S@pjvj`!>PKyQ!Go(q4(ndlMml_K?N>YPEQR0bF8_inSlzf0>Qlao}TEisyBlpehr~n5M_%%y69!ARGoFLLSm$K}wL|i=%>DRzaAycfO73*D(F2G~M}yf)40NPQ@Oq<=C=8fls?`sy;W@Fd%8BrXpoLCEdQ871G4I)x;~_-yWmZ8#iO5NW<4@#t~1H#6-{%1pC5f>I8XI*k{;#I#woLnkAp@QsLD?J83SAQJZx*3t`xTmIdA*jSe+s{-Je%dy#<E2GsKHbQ$$<qpQPxL<0C@ekYe}wrlbF=kb5jecZ}Ri*Q{CV5sfb2i=bNm0snL^Q`5B3~D8byv5tK^ZLzq$KoJ6&*?CxAlj!?YT$uE1b!7|6z5%ukUHII)M_)9k5~}KX?0<z>y41^K#CM)k4z_sZT2!)1ri^E-(apz0f2B%C@WS9Y+f58rv<ggQDC&xkB8WUp6GecWK+7UB?}jD_ArUvw~pu?v9TloN@jPGy*$10K*)R_@3lTiuJ$vK@+JIF2;V>`y1Ae>HTaa65cMV}2#V^e(xx3cbcxX&r)OxW<Jotk=iiCp$rgsNBCXjbn@12K@s7ygJ%(FifPC~N`)Ki)Y8Q1McAiwhi~jh=CBUCm=*hxuXepxa6XB)B0Khi;jORBVn+!T?`K+k$i<bZX>Cgn$5&0<Gmc(0y3~HfbDoTyeS_kph1Q|)F5yDuY4Wbiw1u4HkCl#gs7f8@T$ORGDOI4}CukFUDGmKXyzJ<em0p0)UKnSXBO!IjWr6}j17CDk(jIf5fkUxP?cFX*vP42?MO#-ZWEDV?bHkhV-kS*uzsqE+wA6NZ)pOR_R`k%wq_}s$nH?ig2YB)Q#$)bpJ2GtfI?^065!(2Fw_h9Nfmb{(xSM<X-QUsHulPEhf2$w@!tY$zLPDlGD<A3CIhALg{v6un~fqiVij8`quoh|7jd}9JAO;&j~6^fL3Ma^nzZ_?aTq&4#lgmQ_J$1X=t+-ax<rS~|Vgf+bwinjv!b3iH~c(}0IirT=Gc0vEmXx8UH``YP479#McUy5T}WNuglJMCpa__gp(nbM}VtxhPTcF`b6raB?UjHA+x@Di96{)l5&(x|`!OJtt{x(^}Ow)R0~W754U_?DoVIQAlpSsw5)b|ek)D9C$)j2ic&5s&x5LNl39A-b!640bx~;cwktqL%<QUMH8N)C}`4cCpe`p|td#1Egn+?gYS+(M`xsABgwK4x(s_BY+s3&+|Mt^=_SaH_8;~#c^Y`&sT%&>Oj>2o+Le8F{_91I;>{Y>o%zIzDpWw?!XvtHVCx*cac+Wwp3RL8kkW_xp5hQr^&8-mkW+qIu7fNBjbAzP{lJ6V*>NNs^Y#<Y2`ZnzxB3Sq%nAPmZ=?RSsdp(dMu0a2Njh3feo<i-%SOu^e3Q|FmnG3O^OVo#Pbm=i&-`+%|X60xXe^kcky&#R`Z6dS_$o`(c-joSH38Uf^4YJZ%t8d{qpEb@gqsmYR%@9fKCZgh?BGFL9W6t(KW<aY|iL1-3&Ky(}`taP<%#3-PK~r=Z8yugoQaU0hbz@198Dq+<JGXkhx~%s!>rO8r)*S0Fj92sA&ORG>BN?ZZN=p>(@2ePfGg~qG}NX8f~ngm?v?4j=h;{x7jS!K{Tu<#Ji^24qN^^7n2HB?a0)x?=$PfUz>vZ!}IIgRS4?XccqMgjwyFOPd)No-%k)hqZ5goIiT4*-D7ExWx$cWVve@Y5`ATKenBl0w*L8L()xVvoUNHtIXLK_te-czGGI%%$u0iC5EgzNY-B7mu83YfH#5Q&#0_H009@J$Wayq#l2cZOp=@_%zh4`(k@^_PeU2K8P4CY4)ROg96k%ZT9cffU&CrEAC1QgIg#^-}l{}7-4o0l6I6^yhYD($6?^-f@vurSRtP=klm3Spya=}DNS6ovsF<!nl^pf9iBSG#I02ftc9@e=SQ<#>8(^$%G)9GsWjsA6RRz*~w&1^*zD~ZIMezFB_3fC%201x`l)sK%QWNm@`e5>ChQvch#d4t6>Fct`xz+f!P+=#m(6hui$HXX#7kae0JJCJXoM>mCm5ZEvVAWNtk58hH%)M;s9h+OiBKH$PTUel%NHSD~fv>8D7Ng##f;KzZyxO0SWnq@l$$OEHeTxnP>uk45xAsXumj)nqT6;1os@DcqcspJ_Z!H9W%b(}z%b)fHh+5x;<z9r};2@YKmb(zP?_Bl7HLki8A)JK_y1$u4J)`#O)m5Xw_6r+C16~2IpM}$BFTq}-bH<7(BT?3(R6L&b=>-6DOm}w`Vvg%>}W<XBru%f*PLS`~|tt9&n!94L)iTPzI&O~YG2Rp}b^;4?k_p^r$O#rz8WoQ!SDdGJI8+XAVGQ$}={jS=e?i8%3WDF5<s`8sAJKw2Bu#824x&H&Y747Q6gCPH8Pm(*DFpX~{QO9#vWx82kfFG|VKb56#Sr?#{FMd*?H;ohu38JnT99c~gW{EYqFb=uvu<S7<uon!Q_GN1d5@)K<ecP95unO-o*4W17{ptI?M0_;U&-OWw-4u*0{M0zLv!+L`=dfcJ$-s|~_KPS)y@CfaEJbE>@^KEp6n7X*c!d!)#a)hRTy?p@oBDmuM)~QBQ|M%mI_HV@C;9C@4i0XfN{TwJgM7vPh3HhggHqMsOdDp|p}at|00dSUD6Ufvi`+Y(N-zit`?dVvAo-bk7vc<BBHe4-4_+TlVXqDLyHhow&v?S1tGiU$IK?lWNGSfL2b#l3TNLfs&QdiOGZkmTy>#np!N^-^pdu-vV{Q2EI~jK6*ZZ0{L5CA%v3K><Rs%LG)75<K!a3rwCn#Z5;Rh~2!)Tf2R8)vk|J7>L^T)#tIcMb0*EZtCtNNaR{z_MnN}Sb}%b>w9#z*K99OAlBz^S^(bb&zQg(VH%HZwHEK_=ByJuX2K?q>FWlV`8-$aEN<dN*5QBX+hkOh2EszV`OuC0y4OOZwO)J^BSp-2-0O2ou6PA46FW$Rhn}d!w5iTHVU64VgXqRXa#QX&mm><>+h!{NGHK@c>B~?)Gt{AswW5z5dcHriXa%a>SI9_OKEmi}2AXoF$%&v8)^Ct{0m{lZ$92X7wyPK!Rl-g$LKIOURZUS{c31ts)@@drQ!nDMBT4?Ls#~t#QnOkZ<I@1p6@1?NhJvNtAIu>iL|*ruS;58x~rkl@|)c$io~f%<c{-dFG-xr9c*evia19>?s4F)8V&p{ugQdavqS?lut^D^29Y$I?%in<l)_Jma!T0FylVl1O4V2)1D?C)IReMw;FM!KYIFB_1;+u#nv~-!l@Rs2GY=GAj2cy?)n+f5K3+Jl3_1cG5Rwgouejr(StUrR-n!$8Q)Hb|4iZ@j%O^?mv<ZUiz_s;1JniGh~d4JR>CVdJo&`P=A852tk#b%mGw%{_9;S?5h^l9OEl_%I%EC6#uo$DdaKZpTW7-kSv?`Bl%B52k(5&b82<^@{(*+~h9<g=vgAU8U;ujwLLh`WH6#@qN6eMOnqXBs{m}V<?EPRR3K72hh=<76!{Kc6P8z+TmWF=7wIoubpCxTm!{LzJ0fP)yvj+iwrzD@$&?wji@gN9!BN;X5sXW^bs*yRMj|=-+TQ2hs69s1l3igM?cew%yZLfJ#??z6-<hpgLII$HSx%q$@&Z!Wy-RP@xPROe#*MT})@Q)6o=Uc5c5P&N*T+_y^H}6#!&*?5W9oznas&oF`b`cdS!GPuI+5dvf@F#t>QYwSYEq=AI$g|}Ci&ejLp*8sJbB4KvR4kJ0{G;h?K5h@uLalo{bWkO7)uWt?AlCHnFqF)_H9ZMEqkCrrX+PKA$HM-sh4Q$l{!<1&dj))#?AeOl#hEsuM?&qdt6;I`{_0X51$NErtG4bKiB7j=q9#2*;eGbS$JvD)*M{xifN~q9lOE~frB}#}5=xN2TRY6&SGBEbTpj<xawYS`5JAI|Kq85Cf;?Z7#L(83^24HpaPxhu#h`vmB#&x{Im8imcshcY=YOhVdcVcXGQ}p9`ymAD&hWx=wT``rU42IM{)UYj99YX44hg<0x!ys+Q*T=e!8wb~cYzPi>RIee1xAQ^m`irsvo96{0}S&pS^W+#cM!YXI`P@&ZEm`Gt^=8%67?jP=mZ+L)Yx>9>{3N&{&VS(5|{{Js2yoQ6r{4z4l@{D3*Y`e*}@Pr#EU7+=B@kbFc<<Odsyq%j2OYB-j>Qkr|tf;{u^06vSgghU42fd=~KArQYAHAtS{WYP@H!jKR3IMRqI&sF`BtrrjR3JZ?<`qUJ40+s!QN(kc3ocTc)O0UsrlL@}y)#ZB|&8TOuBO3Af*K7CV7{iYI8^Vo;`W;?-&`ot>epYwWNVPx!2dXy}-NC;xZ*L)>1@9n`rUQj`}0LvqLU@UrQ&!zbgLY(OL0<v3l7=*63rjRbLXUoqbkSeeDq&TQRrujvZ<KykK-R0n$<vW?g%Kw>m8*eXhhW&ASKoRIY=OVHb0xdwHRwa^Zn^f$$2tApO621La5a8O%M%dP1J9pNA8hESlI=jDE=zq92+=1UE{7Pjsepro9yZq+P!y4NkXo=u7-QvE`1!)0R4to?JU#J*qc_R#+K7?7~^3_3QsM{0T=p%pG;0e}cWlBoopPIdKiFyn+yUh@tUow4@nKf-OBInF(vP&>ht7j(ihpgja`r2TG8e#FtwIq6qfrw)MW-TD@SS{s1!^<FY7d;1F}yC$6VoXWYE>0K=Y&TKeJVi51jgAqS^GX_k-`w7~t;IoI4>S=8($x^p3pHqNcFsx-kUoHGAw|+_Aw-5bxf<<mJdI6%9qusa3)GpykxUs1(L=L*1-^(er4GV3jo`bf$w7Pm0?{#$i!w~hfQGE44ZRkJTTT@$0L*M{qP~B>`UJ$yuEVt41?sK*hrQ$5T_${H#%2h}Dk6OQ=Ww>$vAGvk;!Ya(6?JOWCC{H@$PCWgCWdaAENdYl0`glk6VV~?~T75_F{DX8AE4QFT-P}>zkMZ27FG=6_h)j#>!0or2_F|AcUJ?<AJM^fpuw^vG?vkgV9$QAb*E-r%_BDo6FL+B)SSn$7Fy(i6Y20QBZ1Q7%1*4l}fA|t6zxV42Z;5po5XvaOY|Kn(m2kBVnz})jcHA<ao_Fp?YqJv;5w|wb)tgmvWL%oHKJb7@O%zUvDTm?1&T)jY#@<l6tSQ4&r9h}mhQu%Rkh<Nm;3Iatsa*G+B2UMPW*#J6vLGE|CY6FX4H^{{3uR<v%ty)AJ$cL22q5KZv$3{O<0Y_}DQVz9pb7z%8!W$&&1cedi-#b?_t2ffZj^~;D!#vW!}Nke?lWDv*)+FX&L8EW7}=bT95l+y)DmHaR)5Nq+*726q-yol3-L6UncaD<8U@mii`rErD!JT%?C!x-$@9^uQc1wE0zP1f!)TjjFP)!pi(nQ6cwwftvy1R|a<{Limn2VUefla*km(``QWJAbo7pC6{o;T}tM!ju5}n!h$mu%Y3V(X$>Rl_3;-S8tz%OkQN6a{B?*c77R5WC)!xn{ruA4Jk(Xi+HEe12B{0<Z;<w2VN1&dKG$Msoup7sJHC?g5I+_tb<fOd~4)#y+7j!Jdzz|mHZ1<5;ApYO2q*}WpK?X23FcZf5hrco$TN7m{LLbk~FF4JLmH~&d2l}r+tPmQ<k;+}qOM(Dd1_(RkmR(Igv0!EN>m-VxzyD9D&_1eKU`5(x0kcR@ugAST<x$ivO?U%Ts(pZT({IEGi^nsO`rogB+c9$CCUCqE>d^V>g6JocBV0DZo?rg~({NWq3b2|TKRPS9@!kC8$CDKS0YbmZL_xdgjc_Q*rXfy2FUiBg>%LId$WYO<P3ChTJU-W-{2ZJ1=sQ&r}*GQp1==3M$7;8__JJ{$@Bf5}FUsR_IwQ@u<nyqB*-X~DI|GqsQ+C~1T*!?U98o*Ht0tiGZ=n3O>@Ba$0a^kL!FQX9RLr2+Ht!_=tOeb}g{$v)0{3+QoB<fVDI-L3^+%F8NQQ8wzk{p#dL2tjOPa6(CQ%dFLlx&S6No$)2Q}o2YXO6R&m9hv=R#`tTy)7Pc*(T6eQp|z<HSxaQ1=2ikKb{MuAEi4PQnMm8pCEc4s~#LbR*}uJhK@TByv>V?qJvJyWML*Ag6m+^BM2ygQJKp~QK2Uv<$r(mxs8e7^zXGeVp_J2`8DMlf#+aj4UN^=C%i$6!D20Ys<df0PEvu5H`q@?hT7i_K=;D%mXh-T$6f|$%NeSiWe|vEB(5w66ygmUPuyLG6Rp6)1Ls1ZZLIq^@c8kG5<+#QZ1AI50hBO@Y*nee$exE4c@C%^#N=KDNFs=g3B6y|eAF({U03E%rLQ;s=)}hu_up|XIV?7DlJMIMqHjlWon-;66q;@^{&MDL3Vcp<$vc_<<ufgGc;$Zv>FKK!L5S7GP&P+q9K!Gcuz%Q;Kv_+Ro#ANm)0XNeXrS)=H0AZ<cx1HvtAWqSAp=Y`B@?cAoVi{*hpclJ794AajX*>0i5A4kHxqic(4A-bV)f^#VZP2}r(GMT4LPV#jy#H%s2hQvDq4@?)atzOnd1@1cT~jNe?G8;Fj`^=#ufYv9hJ>$9|?sAcDpuMIw`w_(qA}#gkF7rlp4}zaB}3Oe~mAy_bRkZlhS0+9PO?r7>1jPM9i?qq)JZE)dFfjaZ;6?iS5zRu$J=Qq)6N=EgPVTe!}8x;f*nXI}*7&XfC{~9xuapog&+=TVcm!$5XjQ0i#}xho&5TLp4M3LcQYA%c#x5w%duvs&oG&+eLdooo<XJy1VXpj<_9If!*`b;+9*4^(6$oKd`z;BOyv<h<o}Ho?F?@VQ-J}wuqFr{oQ~a^L`n$g(Z!6{0cMQXfnD}cJPz7FnFD$#7}&~aKT$MjmnZr_bv@Msbs77-jm5d1JjTA&9dh!Vcrm*)h~=&4{}Gi#Fi<qonh30g;EI)Z|lDtFQRZ<Pk^=0HE3|E%HAjvnHNL=OX|&V838J@)`c@fo+`cl1fM!h6S*Rwtzm`cw<F7cg}n&6G+PA*F-bP#`}2m@H9T7|pXhQddR?3gFV^=-y#EyuVh}i0Aq{%unI-f@rIGwfAH(__w}6kI4x3FJGb82y4~Nwk16iU0P+4Y@d5C4<t`|h<3Nl~ZFc^#?T81+{=~AC4^t4JYKMs|p!Yoa)$*dJi)`0s7IdJC<V#aljioHLrPk}TK$Esg&;*ToL4kjP8YaOzC5Oiiou`xhzj;oq9rj2vG(jOlKqYQiwUYheZY6^cps=O>lO4EEBlyl^)4COK4V&poq`)@%ljcKquAHt$=hzTx}YNXDa+IK-4)FM7M*0sEsY01RzYDU;PU>Eshc;vQk-usvEg68c1gPueBR7LN@pW4gN_LNQAgu!sgjlxRWIu<miX9EV>EA!{qM(mzobr1({ZEhYAhVk{Rm`*RYY`-dKrhL~btchHWfqV1}WK6i`h*DT;w@T{$lB6JNg8HwE?xux9xKu(DU;v`6+y)f~A$W06pY_#R^OxS@-o)p8E~~RAJMXuWt-p3-1yLD+clL4oChCOoM_bgOhP7VS^ey5ZcbU{OQes8=Zv9<>l28=-2a3jllW6t@W=k`&Y%q9+=fin2k+Z9C+l|i5u~ID*cxb;tcq_*%jdBtANO#*CUCMwMpe0UYa{5C~Z$oij%a&lky-1w$j;^gLeunkwwRlEpLbl0Ej?T|3^{uIsAi)9Y-xK~{%r$s%ud!LM0u#T_TC6m+#DlqS{@>!Ez%x&QgW}vxY~ngSVMW#!GYAijUl&>07RmQ9yJsVTYb$l#@$IgTmn=GxdMLRjVMuKkxIGa6As-MRAVUE(+uO`mJA~F#56~^kx<!KmLe;4`c0ZrSl;dn)emuL@+%7uV6V@xrQyb*UJR{sdlL^IYSwXhT&gh+~qDySLc*;K`!*7lsuGT=ypAsL4(6W2AYs*~P%oFmj*)kH((l65*f7rO6Tvb=b<7HN`4krB2d2)9?%Q62!cB8WxTQHYWr$cTl6ct$f!19K+3~y<tefqB3lk|FA)e@8*P1b6s$L)A4c8NMXOQ)#pX$@%=0pW&1VWyDiKf3j++xoKq3t}7(D#cDkZ`SRiSyX!j`c4uGlbbSm@MWv}g(}!O$)l(r_+X;^z3~$4R8mhzxEfVyZ12=NF5kmz)B@A{pXufdQlDk~j6Mv1iTEI&2c=vI`Q>y@&9AWFb$B(O&;i`$vFRQOpr8$8+BukD=A%eGq@B9#dr<El^+BMAb4*YuB@ZTsYMlDg&_kLhd3Txj%I-WPkhTMNcnZ?~35^l*8eIvCG~n{o6j%1$7?qU)2g7k5DGFOPF(^2P&`if#f+JqwunbZh27clqBU)%{9P)}vZX>?q4ASAKB7MrLA^HE*ee2aJBx4ccfCBmyKK-e_4XVbmOyc%G70m|!wf6+*);$Xo(4k+xl6t;1)wLf7K_mp;LMpA^plr%v$K<oN+thT=9z45hIK>sKi=q`iF`5*JBD=|Fd-c+jOxPw<UPC4W+~%Brw$;-MjJ%S3T(=CXU!AP7#M=S!IKn7q)167=x94m!%adYY(^G~*A(*x2s&7-;Pm*x%-iv_3s*9wxSL7G_@j0Jz`&Shr4ORxkfXYakANTuz?$)GqKXZm=8X<g(yK{<;C$p0!4WR2y^1ZVX-x_o@LQI?uy|wh@H8_SwG#xYbRfd)g-_@^nEv~pOz9Ry4!!Tk+)-RaeVp|!hh`kcNdA-`kRi-^ebNsBd(70!3v;wEwpykmAqBMta@fu-&bGg4Uv^2V_$q@Z;a)QWx+FG#u?d$6!Q$YOKOtWY0W%o?>VS5QYF5V=4DckQCKR8sZvr&&F&UP?8VK05AfKAqNB{GVjOUnlzM}Z~}CnCb?b%FEtyc{f#DBYXD>ZDn&Djr7{D~0-0QuS?t$=7GU<PTJB8S;C5Nij_$+^J*gv=8-={X~`RcEd@ot0w*IDsQlklq}x^R+X6r4)I|VVvpu<FMnmgQk-pK$8+#LGCg|1#Y{Ng>+HJ%xE%$_K#NtvN?5ah=O5#K6oz2qLp*meTl`hpI0W#qHN3CCjLx7;#ntKUYE>bGz@(kk1pZZ=jvUD;dQkd->))TQgq<w}K3B5IrTkXHVdC!pA9&lR{Gw->ieW(sb_N(b(AK|+%%nK%cSx{r=21{z7(LsNhesA&8pZClM5(%TPD`K<pIs-wKo?&qEn`k_D)Uz$Ax#p2WS!cZR}IV&VJ}u7>G9*Fx;EaNctE^I@!vD@pCE`-z=er0nQgwpZLq{}He7@qUISCqxstFSZ}*O;(vmrqWeZ_YZIZ|{si4Sx?zp9X+&$?~{IO!F1!Um1-b5|lhF#lUU!{-~aDYNnY%ut}<jDGK_id4)S(uO`z>E1g4?{vURNNZr0$jZ;)Lo`CCvV)zfYjN`6Z6g!&9~+8_$~57{7+!!0i-~mgK)Bw>SsbAh{|71MnTo$382;4a=ds3KE7>NT6AF+FiQ6#4C!7T{J{`tvBYyu5^LN{Nj2Yo65~t!n`4E&3S?X%^GNdP=W2F1dJhK1n>wwhwifC<#aPrIjCY34X542}43YP}0OEX9W2SqzCJIz@ln6d}u!?kfW0OBwk&&wx$A}xxv<?iSTciv^-<?&j3vrn8eo7<2%-9zrKt~*wu4KE_*b~xLuku4sOEcJBE~(=&9L?D&%iLu}oI=7G*J4w&36pIg#Un@W`O9H=a3z(daENB^o}1Z!^|I<CuhaMZw-fZVE!3DvRUzPSZ=jO)K5g}d-)3kTnTZv#m`{jhuCgdMhCfa-XMPixFL@bVw3E=Av~LjG6rv+T4Cf-^Q#a(cv3MQ!dqz+28#(yoH~fcXLJEzx*?Ptpg@l#Mrb7m%g%^l?Icb&XevWRr{9$%GMno=O#4vcX*%#4<9Hi9vAfy4hnOA!E_9J^AliR|D8E+#uE~iX*HMH(gywvjt`MO>$x=DWe5z5^QarwRd-_%^aA^f`Z5xW2Q9^(8?f-y`nL-z^4p-=YNe-8P_&~`on4&H{i8-hp{+QJx(m)$~6pew>LAwIHO^c!jn-uhVPqph+P`MdzUelfC4;8?Hv?Gi?@<n-q{5_v5XD(!j~piexaf3J8y9^X+fG?NdGP(3tVY6{>N?b&V{t2dp{%9h4FXdN7TFlwv{uaLV5k^a|^OAC=TUS_yKpSo*2{VjQ|eOtQgd>ie9c2gxvU(W<VnCaPoNvKy?x<DYhs8I!)F>)g_RfLI+S#>-mCl5L<2EkFjCG~CK#&i@x5?<h6Wl*z>4JISg__H7dJbWiCbSB2TGbAHTlO;LOzL1#{{Z8|F%!2ps?95vT3F}7N<k46(os27$A*SgH(|NN}y&0Z2nEG@zm~^shBJ53;dUtod|A+!@6V~OAa{rh(Jt?M2$l(Jf^HHYFyv19MU#WF^yi=>XR!bsg+xT6xkJ6o@1irLKuI{9I%jN&f<TnJw(@W^~y7lh%9HQMyYgPA%@9`&%{sb{Q1m%>ydCr>)+uUK^-fAn{+64{LbeyVO;qp|NhUC^3CVJCck^fvwnupVfwXG>tY|F@ESpi0fVxO)EC*q^U;3^ORn8=qU%sOwOz;(3U51JcTyw9A0P9v;wf%?EMrM-vpunf<J?=b+JB=C@Xtn80prz@;UnybX&?Y2<9^x<4SOY~F{2vG`^rZ_&wF<803@v#gjidCy*PnpX<sv@kXRr*#EIAN9k9lk5zr63EB`aCEGGb<^D@2NqSr5~Ik)Mba|a0^dA1!O4*u+66fZt}|FYvb;mPdgZgqh{0;W+!v&$aOgbEOXP^J6K`uxFyz|t??2?9cVcpou;_rr!3+%g%`L{aBEX4Y7Eyj4}EMg9-&*w;j3}ye6;{knn@oWgLC0*AA38ab8~n*=|DPPRIc3N7@$Cm#1%S-kZN4D^(s4UQO_9I50T~vQaNB&eQQIU_9VT-?3!MQqk8c1={U&(!k5t%0v9=5sHGzy+9Aujw-W^c(|%wXN!aXJg3-V}&dkTa>&vn+h9g<_CW7D#THn?(aZBCHTNG8NP_{1XNuPEDneAyvmtsVfWtNw6TF?amVSqaqG7<k5G2C~DThFNN3M#&o9c#LXB_^Djb4pIf#(|WULC4P5Gy3jHa(dB6yqXI@>uQ?-Wfw^)t$1L6ta8YW{o`HbG?<|%H3Uf4{%xAIuDuVKQjSYn8jf_!lnxxA`d0S!y8H|;?Y}HcKX?-9LjYhTXF1v~ie7am@b~GnhIS11#^XNl%sYKkn25V=vaEV2e2!$_!_OF3I~j)QC-nz@{wI9KY5)B5=KJ$31bEi)xzP#9W+pQ-MJIea%`xJSM8k0-PVeL@uc>Vj%6-;|**7aFF-akI{shHG3dS)N&4)#6*jFht2l7SF3#K^~8efuM`ZG?05;c*G(D`F+<MmX4q-VHUcT-opwtS<0>pRLpL{K4)W3VmdO3FbY(A<+iv$b$7X?yI|_VOZ11omr9X$u?aY6$m4z8@Twi22;Se+YWB?Dra&SZ_LBESl*Qeh(}3@n8m*kdgvP30K%q6$DFHK@OLM^DOAxXHFr4gW{tpLyT5B7H6$yZq>FRbkxSEEkPf_#qm09vIkQ*)~XY`DZH|#Vn+^}_Ty&n@)-XB+~&|^I8Js|J?2QZq%e%7LxS3Kx2SNHo62VR!QxMMM7Vh&$udKeW7*cK4jgRx2|UUg>61T<_Mg1OJ<XkA@>l6Q&91VGylO#KoW8zChhb>ciHrabTjx-8B@)eE8q0xyw@USo?ujL}+}r7UFJGDs-xX6dql&%3fLvPSx~Ck6L#!ad@<_-M!94_u<K*FW&+(OEO7HG9e@N#YR~0&>#GAH}Z*8wo(E08B4UfY+;~ar)j8_z846Prr*=%&$f6FW3uN5-CdRRpwI!}IlX5i_;_*-An=W(u`IHb*ZJD@-_4$??7|1MxdCrSaJkHG1TBGHg>!gnXpwN)G8>x1G(Osw+-F=KTTR$hl1lwZd#-eCO9F9SHZ<!Tb7NCdx;Jnf=}q)Pm51MsJk5IO<&pm@3f;+J%)=@A5bQ$pRF1a4iih*ww7e;*ZWu5zQ^4#XS!#xE&3>n1!hA#Eh|wBOlgPmNz98|akM79<ZzNF@8j*RB0|eQUxsIZb_8XebMQIGUu<otN-12N;3~&&Bt4T_wrJoyDR2UcsU@`)RsO7(|77Qv9ajyI2UuMitEjwgrSKB!Q6L2sIXJ;SrG=3-Z0UV&j`{pL)7s7H``$Pw%AE7|_g~9DqciCmc>B5uTq{Z7H``TCZ9V@wUTl>s=L1wYI2C*_(C2_Ye?kCqkaf(0C>lfoEmpzgfLS5$+B_IWLPAXfMc1b?-Y!Vqwl6z7X5F(A{!gHC3@->gYt&O>J9##jviNO1%`3HkL=G=*{6Xbe}o7DMN%69kFH2b8y1#oVvIoP=wms@yB^4FMi(8`Z|w!fT<b}#Xlcn|0Hw$!1ud7Va%#q&2Q+_)=~uP2I0XG?Puu=pL;p>bm0X8Lzh-8YY=QNJFdBc=C|~(oC6oM%{2(3!Xx`{zLpnm(H=)JyyqcVGc+E5owTDZFcS*e@?K1H-=mWH76;}av#~<dFwCVbU?L}7o2&RD1pH^Oa}M@|3-19NyfeFhk6>=f^ThP=2jSs~$xCMb-X&}tv$w>wZ(WuOXODty-|rtntaj&a;99Bw1#(Bw12{+rd0<zgsS_<SpuBc_cfXLCqA_2pCr;y~Sl27d=DwpN&#!Gm&ggEHG<L_O--^7fHdjn&9Qt0MXkGB+>Fh!m%wgh(rA+<9*PIRN--zf`sHAmg#OB}po6cz=p3guiM;}jU6rihqfNcIu7Pc=>_VQ|e8!qpzQ<{jOySUSgz1l%tHs68%*&7a+xkB>#(;U0-t(&na&W9j3;b+D~{N5>u3ZLy#>Up7c(zRBUtRQ4dmnN}Lek_Y}<CY~{CIqjmf1orT&9gqmN3qWxuQ|}Mt%I&Xpfum%5B&;uG(oLI;{^bV<I4W-6PCm)O7WsaS_x_6K;e(q^N4nbR(d$>a;)%i!;2;TjqgbkNpuIyBF-<E3v=su`_owLtQwDX?ew~UhM%ji3Dd*5nc}x{Y3Xzs!{#4U>Nd71RmG$%I$o^(NsGskb;uH?g5%`Eippyp;KS|r4m20Tz9#Pk$IQK?hqvI#2GR)%Jp~zG+ZvSG$b%6ghbIa)AxWfw{LSK9>!mo%$p7{$F`bar0`mK-yl|nt>LZ`Ni48>Liqc*SHKxbY`waG@xSy>IL4>FGOcX6AHYWi0l;}M0IFb2yjp~(xEeq>yN1Xq#9AZ`YFKAZXJe<1)z|Uw}O4NIJ>PIV55-v}!5*C>d(lt9mdL}x01~9>P>Ljxo)9`ZA4j~I^IC-i!@B=Fq5@4W;q1={iK3eVv*$^Yo=6#(L7{yqRfA#b<Oz5zD<u*tikGPj&^*OMnL}{*AWY^hUW5iqB`i}6!I<3?$RIs%?A>_V=PTqi!y+2^3zlE=sG9KwGD!F!{6QZ>jZbC{?5l}yoS!n}|Q}%gOZ6TFv4+B4Q`aILoE-P@qJTn(T&#40^S794i-H*deRJMEzeyZZY#M0r)e(#e(t+0&cVV7jUi5xiV8l@SPYqp=|gh^Svts+4j9G2YXaFH3a=6*1iH6}_>>x$3o|A>5i>(@x!djvMXt6#ozO4b2?lWw2lS(i&S2p2z71PxE%oA@0AFD;W9n4r7}mK+d24kb##6gPKFA4{4OVykc;o>UST6@o##cnK^*YD=vCBwWM_f~K(rQRHs;@C{w<jbch<Obz4wH@|TftFF-hbM^ZgsZyhSAiNNI$ReyOWDEWD)D$9!31@M|0n_uK@(CQryHn>)KEWpWW0CtnaBO=O!N^+AQt%Gk4(to4ABjs1#LV|3nlh6Li~<6OwW*R&nI4cq_b0XeX7Kwr0f@2pMm1CKxyowkm{%pCN&w6w&07JDh>*JFqyG-Bt;^RO6nTlwZ#A^v8q6eJ05UhL5CI}`;JyyUtp1gV`H(_A9iO^%eI!CL$Jg>*woC2UCk$G5fDN_tc8V6h1{=suMMWUwUKL+t`z4GHCc?-tp(%{~b&Rgxkb2dB={uyttuT17{C-E3tc%HQ2m!wDcf78j1xx?TJCXohO5ZyFjawYcguOGJ2g^pb%|i=Ycvq!a1KSLC!d{#&tB2{*c%ERvwEK#aT9@pv@^zk1(ig8tE^hOXe;7ljA|x^g8#KR#`uhlk*4ItmnbpnhHt+Cfcj$^jQV<-h1m~KXWn+PAT#h2xS{~!lf@vp`INB0WOsJ;JaOe+in8co`6HBSDiKQ+ntl@);8`7&iuK>@5XQlT*U6&I_vdDZ?F+T|e8w*wvHQ>+Vxk%s7)vyI8*T40i*=E?c<4uE!JPvhjYFvAF9fX}b`~j+XFA(4|%z5X|aIU@hhO1`zXkHOdC(iLO$;sRs(7lZaqKimE_F}HW-^+)I)8Y4Y&h0=-v09RBF%*rqEeRFu|JWgUCxEgC=<?fK`f$W^n2876*tm&@aeY$tTW{-(4Oz=ADHJ{*qvfc+)c2c*cIhn5<NRZ+JVdMvzz?56%;-O@W6|M;b#O<w28h%}<aD>-(>!ZG1lO?xz2Fhh{TBTJmA$kX=#PcoDWOS^MK2C(V|9AuJDUH6*gNPr(WF7ux3wgo`+d6v%Bk+>tDa4uf^X+WDXh~rocQ@TxVoG*t2Dqkn>9#bjZdWU$Q5<%-8a$@`NrAY+{fV5ho%*EHsqPvb&-(s6LUv+CFJ9{A_f;CS#cD~r7VC@oUT_kE6y}zc}bSCms0^bT*+T~ZgCc78&)rDp{?}W@x`PP*Kmo}3CMlLU-%T2t0&_4>isMqzB@-ky4Wl9_w?KTz<!bS{JV5r@UJ*w+K0iI0WHFdQ_fIg(E+kKKI74GQ*C2By^8y!I4+v8H$ne$*LRQNK{s^)RRJ#Gk&mh`0V9L$T_eHPfVndr7}^f6ii;^NJ0G3DClguX6$O<h8|iB`T-(LEOcrht8@yTF_I)wF*wD4tz<0`$-l;VxjgcMchp86jG|<!!*`fR9@U03eC+F<gv08U2_wL{(*%;Gx?ft-THkDEO?81cW7`TXb@JqP(mG?3if>e7SJKP1?=$~{}xDdhFmL5)F50_?qIeLbmZZwgOmP9PH@1blwNac=d25qZUuQnH^h`42#=Q;Q=AMocz$k~yW?zHHbPO;p1bQz$pOuldSl{{}jH^2vRaascU;|;(nZy5|m!+LdBId07G8Za)hdRu+p45*ckKqRnD;^a}pB+n?!I4mvoflkw{R#yBNJLvZF`-ckXAlyt2-4opAF5)6L?1^W`)50#tb(40$p0I`Onb&i2T8|43p-807xp8{P!V}FMA1!Pi!xwX0es7^L);cp4y@L(Tf_$8u(ow$P(zS|mm5%sG)6>eUBPpPUV6V9wNN_;EPD@UnJnjCE_za^gU;TN+<Gg#7c%){a<3vNe1sP>@clQO|lAy&0x|gI^Nr@y^Ov8C|&gE$d6-PGwGaZw7Arz9cpW7>ID>%MD`y&rT44Y29FtjPG>RZh`nh#rv66*ESchSuGu`p*z?3*m5aQko~Zk&qla$f#iKdLCU$ppr!h|=W3hZLZIp0WrUAh6o4uQ6Q+=xm_n8%rak><@H70>{lHw`SSR>PiW`yZ}3q5%wg{LnrQQfm+U>#0Se1iJt;9v#Smo6QWpCzu3uMM-uQ~Th0?z6Lz=)f<WDnyvCm3MKX)bdLf~Bu%<NrEe1+t&yX?XKON8g3k~-{Jokf*ejw@L;mx>Kl;4ze17XmCHcB#|Q#LC5|5}8&c`k)IJz$i{lS=MJmdoM1_>XsCd5WT?-3?in-=5B<87ZGtlfz=+<^d;T@QV{M{$MGf0TaFx7=Hz?r%4!DsMLT9w;_T2J%xsWL@P^GIM7h+N=xRNpTcmr&haiQMOJDn>u3T9MN}%q*jN}IzY&Mmk&L5mWUP@!q_(=>hC9$(f|v*Y6UC5n%I8kt@k#_dG=IIZJju1_p}j%7LZ7O`0y-qt+hS@))9LUv-OZIlDn%nA*+ngiW{>b)V)4gQ5)Kmo^enN?`FK%z+)lkOjPSiPB?qLZf{XMeAa4n~Cr*oKXFv&@#=HLaK}uU~`Mk^))FiEoy@!Qalm=x^X3}&AWQL&e`=D`nSgvKxJlk{`-aVft9(2!hyJg0yL$UZ%Z*lwJZc82rjhRHY?UYm<8X-~)5u#V+P`dl%b{N)nxzgUw3W%pR!CgC5hZ5xTNrlYk0LE3bvMN<l@D~0iBRfr(jjRa|f!wttLfL)}h|PGH5{kF2rV`%HFZuzmi`Vn?(!04UhQ=p@w6Tt?>?#)SVT1-cbKU?q-a1MxWWXsmjAT5g<Aj^k|G0TK5JJN|$Z-~-Lt{(|XxhTU$p5@G!YygxEfP}hw<@u%39=RVvg34$Ib;yrZi@$%C;6Hhv=W<yRr2yKao1!yH}nSLqvDwO*lZ<rtBxe^KpRy%LzqF2`j~bt+Y{Sr55pZ5agQ8Lmh$2$>e`9jYTF>se%%h2btP5?aB1>8Y*P7R@+h=IeS7hxbdm}#?#a1j+F<EvshP~Jr?x|tXs@u1r_L*gGLD%P86Y*kqu*}GMQ4(9Rp=*HH9LZ5-x}grhKn{cl_!p<2X1sIu!ffwFm{ZTg#HTr=p1-L#Xt7%fc8x(BuF>Pxv+r@q{I@2{BXm1XTtIE+dicGps#yWts!lz{hb|Y7xRw7Oc}1#3wJ4Q2CT36`a=sX%nuS;;CG!l<8#IC?J=o{K%7dwaW4vYhYAr@SB@RI6t}9~XbnbCK(}ciDTE?Oqrue81^et;!PQ`>yPFz)BrN){un|J!83|iBpFnIRy3Zg1cW>ZI{sJGv?~+`})e_uSQo>C1LwqX7CO*VoC39t)Hw!S0^~#kz%Ll=SPk{1oj4?L(=M_dI#7qEO0vE9weEe2UL^$<N)ZnASPY@>9lte(0-G8CA=+fJ-31D|SX%MB>Zk4d4){1Edp4%0B@phvk%#r9;kd2ET6YcP|R;1l@0kGvbv`W*s5@{>y@J9dr%}ti^oHGVT$faTh_BaII5Ene~{(qud1-(2|{?QsvQC7CO?-?zP;1vUOwW)j3*R0@a0lXj&GZdqr<s}&oIFm{9l)|?rSu|_P5o3vEFUenLAyb(0Z?$#z(WK)lPs|R~#L*+vo~)C!a2Eqe(7Z^=gSvRan9KjDLt6KU2t*@kuf~u1cyW@ZX{mOdCR2HpO#g4Fa<N+6A<mv37y-(Twz~toJ->M?6$Z}E^J3+H;w!%G$ZVWTl6bc|vu<C4<V&9i&WP_FObKIGAC4}C>RH7<CG4M$uJ>2@DmUG;mfRR2NJB$iG%v?v1YA)MNe1IJde2C+E{*R9UzRLmF6oa1r()RVW5`772gi}zfwD&=gWmTDMf-#%ffwqwCUdPdISbI;v$Y@CHUFka!wI#MpOXFcu3}Tzm?FCe!BsmC#gaaRs<!6zK2~}^5;UR(nxr+#Yi~Fxi+v6|<FWy~z_gnI@?>XZxN+_=mNw-R&*`Ozx=`3S<42T>l@YC1j&rKOJIjv9Eb`yLf)(UM4_SXVSwi1C6z|p|kffJI5}htO>MG*hTbGj4PePm*$AiGCCNVZyl+MNe&bk4Gs`*1yJR5mP)odly!iF-r6!?L*(0>u#<}yP-ji+f25W|@Yf_Qe0fjX1c9(qY+O^jMA8+ndpLwg^WK?}y+HvmLU5r<|qw2|uVVnStO$~*jp1uFCeIEEz8evq1+w1uh(Oz^)X^Zd`RI(AM8hY-n1_D3?JqRK2@4aa;>?b#ENsQ?g6_lnK_Y4QB&2M&6N>MV;WaGtv^?sL2_9cgE3L+P3%!ee&!Y;YKi!6rdk-ur<g8_O}JCUH+m=h8N2icD-sJMbX&G@wYG)zt16Y9x4)PNVYh+Ts#z)dlmz#@U;sgoXZexZ~W}+UXZNmCmv%PSNCczmn(M+^ZERpctIf3V>ISeK%lKyHp=OD0JxPiBjW56lL5^SoPiZ2|j7DW8Yl2x{h0WD?l%F%+UsF?(V2hVZ-_3P3%WP_vEy7=oCh%Pu2Ch2f~hG|17-9y%EK}XQDke^$Vfr2ncIXTYcIwqaPHK7lWi5`{$srDx;81NHEH`{WD%kim>s~i-ZG-0*ZcM#q{o>HcP4CZ7DE;(EiYJ^nMf;p!t2g8Nyx=BQpna!xG8F=1}y&UE=p9Gf?$D_$hj~l`!$t(6+fJBGef_*=>Xx4N`qP<X6ankwtLyhv;P=Tpl%bXX`X=kDp>}aLoPq>6*6Qx&gx%6X<dLu;&VA6VUzC8a!db;O7_L<HBN@M%Mc%ejij_ScBlQ`#S(SS2t{%*=Ty0o+JNDuT8N=4L?(EFuKnL4=nw0J%%|vsgz~A)cnYf<`NG!CmO@Snp|<r<6H2c4#9KNQuwA8=jh$h>WFCo$P>CKwN!}@TkOGczV5qmD91H$SqC)}nCyi-yXDJE`QxIz#XUFoOQLG+5Z8e&iMc`0!&sxW<hcvP0PRCqa<P!V8G6wJZC5u^=9gfUW1)vZ8Gu|jyF*%G{J;ebe5`5NM<Qzxi>FD#f@U{swO*Fs(`2-DhmVoiZ-J~=++N|V0Pnh$4l$6#kN%0KKQ5O=sa#83c{QNzfEuxt+sS>+UT*kmA%aaNAK(&f{yEmdBMP17qGl-4^dRF?h~Jk{#Y}a@xbq3YnBR-GruXy`E5XspbUbRh^86>L(sgeqeV#P|48D6-919_+BQ@8;>U#_FZ5m1OjtobNE_U4|3XBo{IEqTDrqeGI%{3!mUF#6TMiPVQ#MN+zAvNHy`R>@U($W<7PVH+erZvJN6ouPU?wwLbV}RoqTGzlzgi$Hx8?4tI;TBy`-GORAm_q4}zF#8PSPZidQ0MlY=rYI6D6>rcJE;h^K8NnAwv{Ie7|#1nW?~2KM@jM0H2=OcoDBHl<>gnKfNjOc!_=j&EYA8&670j|WqI43Zad_TOxiK-C|gd+W-y6#SRd%h8-1t~xzk#nCPm^_&6{ekN^vXX|I~Y{u$P}B_G{{ep1Nc+>ENhw+v5^wwvlF&{{Fi8w5veL1BfkPM0@D9%-;jp*p3Vcf7+tqDA=wsdX(-|^8P;7fOu6oR30tylaE#^+>EV?!3I0@p-SP8*kI*U0n*yI<UZzCl+23U-YYYsCnwSPGYk$$w_h}KTVgy?3(Gfh00Vs#`M~Mx25e;ro)wpk6*XZaUjWVpr=_J}aAQ5P-b|Dl0|Y|w)VFI>O?Tz|8Bi_%;_H9r-Ryd#Z(jlise0=G@?>W@HfjI#!tm@O1&MKCyTMy;xL_xM<B{vrR9$g00!@|S07gLwy1O5e-$6Ma#%om;0l;ka#st(+V&D5<{vkaX&13~Gw7T6jb?8#qV91N~B#ehJ4^aX=hRgG-&}l!xPiOXP@1u@Ctq@MEP;+babTGfIeyU}peJ-Mk$UQhQ4d*r-5gEu}1GWdkM>2M2qY8vu&Zm#Z>B0OkiTN;g^*q)9odxdJC?|ap^rJBJV>eN%AWzR{(p|4Cc5c7fj)XXr^?O}rZ4A7Y*yq`FpErhqE$$iWg~fbpcSL%Fglnwl4T!mS<0g^d+M2sW^`HAbr=7AfSv{ZzW4$&!TnHu(SE?0GmYJu5)TJmlwIs>Cv2~U3=#P#!VZE5PPiON3M`2FsmyQlWVg_+;C%#mri(eHDAw6rY&DouA2z`<W_X<}}t0;bIOM$oe`gbABt@+`B0g9bxAPs+2PEJ8{MsbIaa+ea?-!wsW3?^2L$w<Ds+6dNZ(8HYGhoFycl$gJr9b*lck7-<OG${_?*?H$QI2Wa`_L)%10!e?)oJ>YJ;TV}awn{+`E3yLaK5B2<YLbwxklPnJ@#v8eq+u4yEymtDoj)8_8<aFRSeli>eTHrlML=?%3bK&9n7DnqWuxrHI!yO$a=fO4m+FPs?An|z5}M@K*2W2~y40KNZ~FJQZy&Bp@NY>T?F>g2UG^|M6&qS0NXW3s!<7bz)0yYmJn*}GMii*O*eW#TeVN(WId>%dAQA_Q(P$a|X_pe3M;N0w`%+swg_kCr@OD9tbPa$1gHflM+}((UX|3@0YJ}#n8VUFSmvnRQ@IjlPpW)@O;)3l~gn}EQtSrp>ZbFjP@R1HMYLYP}+!*&jm5a8~^R_|PNy*%Sx^ojkk2w?U<=lJfYl|<fIWa?mZrJw%E!hxhk|SuTxv~GELtfW9vJ0VDYF$mHIe<F?7g5smCI+E)v9I89-t-d&G^-%g+ZEDb)}W&WOkBeBmqkgr;Ru8fU#fqwLOxfJF_rI?MGwyzAYzc!;K*g$dh#&ykdA9hrJ+Fm<P};(zf3AIUuKW1a=zc9>(r<!OBs%qiC7289IQB#zti`KoU)5oYj40a8;)0bPzK^p1($&%RzBzVvg*t1=<PWjR^}6;DFloV&_jU?64nB3vI}Csv2a??j-d7mU66E;IqIpuwZBggX1usa^y=^W!YpiTz_6q@%g||%jiV<Z7oFAyI(`lpP06Ee<9sMSS9USW?j*zGJ7mXtV_2jP3SHDm_#Ki<jZmN7z5M)I_yvnGO%?Ru6z3+p7S91^BjbSjMv<O#8X#NTNY4ifuMuz@5jFAmKISSw+l{J-HBGD{Fm{SZ2xs%7O=#E%i8Iup5v*=~`0FmxHf*8Pd?*Eh@8@e_v~e?I*_{|~c+5G&P&PnzIvqf~spQF<Oa5$m2^|JeJMT1E(k^UuDFvGubxIk8U&{G%%PL$cDuMUDVn*!O(>0oCLRzz_jn-0OK(sgPz^$=KBRe^gxyrPa!|t)I=&M=%N_1_!;3z!t4DUE^Y5{Nl8mOUEBB$4+*<`5Z!sP-XeoWp)aD;pAp_kP)#exu?K2Qo(Rw)06w*=NsUCO=ourx&rdH^!xT=80Hi^jtOF$`M;-vmP?=DTll!w8LU!{UN=BSDd^5oLRrEzt8m4m_yZV(1VtOTnv1;*1v6wRN}h;GIOktggRs)Z?PF(n6IP;>Rwt6Cs3wVr2B&?=n?tKV2l|plACP8l#XPWqxaJ!}N)7yL>4glE900qnU)fHQxBLI|E!`8(i_x#w&&5p;;p|b(s8#Ov{*grFX;)R{BSdh?G|1^ckYSCNo-e#Y`kACcuci(j5o_4*v1%z9PBbwv&^QVY`@*5CsOECVSE_O-AZ?1C5JV*18P2dvlRmi54_(;n2zT{zYyN_kxRJgrpAM-^oOIiRKFRW!=e6*jcyTBA1d7vV1)1WCxZ<<qspwu7H~F#(yMN$%rr~Eg%#d(CF?YIiQhm=tZccrSr!?R<|;7L29-)Hw6-mhj&pbsT2DC00*T&R{Fnl-sB~Z+%-QuroI;E!aDif+-ABP30nxkpkLGEPRW3sS>`>F)IUb`#$7HVs#{3(%KYt!=(%f)L?q~sv6QH#sb=K8OO}GT4^+u3{1==!uM(Dfd_?CLZ0n9KS&CBlnJte80JmgA-7GUIE=_8j;m-Q3XDrinLd~Jwua?Esj(Bk93x@iVF_4x%bVS6I<)xw%1$?W>xI^H#Kq84|36l1$%VvutrEJHC23t0~`(5AhccHmr4hgm*!2(k?{PYf-ceqoop~A!!T7}^BgfVsp*EM+~D)aDF#$fclit&AU&?>&objCl=o{B}5w(mWNr&G1hDTZth=;ap*_WQVNKguTMebI-;0W|ic$c&fiuC-{bTHG(e?zH3u$<u9G+fxaCdHmM?Z_FW0;DUrxGXq~dX;2n+VZt|SYTgh)%zL9qOXvTurT%9MWFWm{)r(!_u1Ps?aovR`c+1i(a}9JHf?^X8q$#bAaM+^nnuy8;4Dp<}m#m_|#E}EbEH)J$BZOTWR}Q`8ob>@=>wy-WFK!V{@z`Rn5EO|v7b5>lU{b7A8#j9I-3m{A*SVKIs8G3r3zKAUh~BTc68cx$(B4oY0$$qO`c<(_j7#*`veZ~FoC3qKeQP;P!s}O}@5n~tD1xTVs|GA~M+R3v;7<1$dlpjUzk3XuT!v3CF*-I>oSZ%%J-XFFl6T_ke+a}4$G)%5O;^G|gop&=c1;k<8Y#jm5d!k~)={e&LvvAQyYiv6;RCWn_>~82Rl|{Vb63>{1)Lmris(Sv!D|I5x+EybR#Dd=6$|3;(CLAA$W0O*g2}W(O5k}BM^p-#s!89g3SEGc%1gmRFc=zu`x3nM@YGh3Q!xd%$F~f&796NomZyLoJ4V!VcuMMrIzZ>%28%lT`H6mq!hrrjiH`$oFfjNR;(n5e@Ja)BWi^eL7<uQe1bLwTvQ0~a9o8m0bu?2S!(1FlG7L<?+2lNY69Arbl}Ri2{QoD$YYqnMmQhQHFhX0&PfuC{J~YR3nh)0oZZ5faW`MBmDNS~w>-awV);zi=Z9WQFgLw56jg$m$L|>(Mo3yBd;ky9XVCHAT#BHtieBhX;BOvt@(^@ESGFKD<Z;GQdhf6Sz@?)ZF^129_vrmjwhF-e1$m9j|KZ1}ah>h_~QsE9GVO#KUjz((^1#fQk9$E!)`&2%U?sMXcvmAf&sTxYTQUlWQC2Pfe)<W?WsN8ya>%&5Ig?fA|wjz^uye~!@98<5dV`|UX=N0x2r(oJW<q&tFYYXF<4g;a0R<WCe-s-cwbuQ@B5Qp4}L>T+jOjR|o)hKp}B5HN@De+T`=cNzesdU>56l}Zp@+(^8({u_A1(RIstnO6B7*3E;T)-{M&j(VJMn9g^Z;Rq&C;X1>ql%l0gdni31U+M)WL1WrZE$6b3?O|YYK;8ymMr5>a{mF;Z9+afyIYqu$fl6Ld!`E~DV6blq!pD0Z%&!#QCtRORxzb0y<tjju($9Q8bBl`S<Xk&c1txOiqr5ysTT%av<mBp?O^Y>`?-bRjU=~F55G4}$-}DHevuuq7skUGG^%VoEyr#{XZ@Jm-G<uX_F}o&{i$-j2d$K>3~c*|Gim7ph5r<Czhp4#q0_1Y^QwJ5v=#m~409Z6iy^-*98plZ8^J!=y#A$B4jl39$tQW^Hd>?b2$R}_qp#>z8%cZYI-}Mo=-3Fg@H6sB<BGcrn@wXhK#?7mEg+zDKOce$9xS>Bvhx~RtW<bbGlJnRfW-Y60z@(BWNEj{BZGNUeL|Ahqj;Lio8sZa%{wgzDwTr9C=yh*O7Kjxd4JPgu7P4Oje^t^rKvpCt1zE93ZWp@@+J~M#An$dqnE-zUp#C<jYYDo%&L-7A{xW#uicK;a$M0ghf^!hbKPTB+{)3PlC|EUtdW<gP<(wTnE=-SvD!+{Z9xI7-$kVkk2D6_>LO3t8Ke|{w~BQF7V8?aFyT1DZL_?NcD7(y=yb><S~pqWW>t>)E!qxQTSkc3|5WA5WFE(btPiVs&&mtw8c0GL9pvwD`J90?6<INToC@BAeD-$-z-D`z1wj;Jb>AxggQ*<HxQj~dW2Ye#mW>Hug3~sw=|77D6-Wp&KDLT_7Z2m<>ET|(It!0P162biCRZGR967LJ{n51aX>!ArpMLOrsE+uFV!F3*1qP5l5W#Gj$s(~tN?gP^#*{ZO13Pl4`+->u?1^VgG@ILUohaBB?DqClCHK25q0q|mgQv%w<33ckSMiJ9v6dGMb)k?Bsz5E0L{?~^saG5e1X0EgYEJ6*xa2}|xu$*X1^YGms<D;SZ|Y?u5_{WhwGTBTQ~SG*5I{?7Q4;zoV`|UZtzm9ra?O9Ha8P6DWe$!)+4)pya)nZ$elqbA;*b%l9c>(MCS(i2;IAHXdNyIIxw<rj>u#mzXE1`5uYCcd70`I_l~&PyA|A)>Ae5|$EA!TRXgN^)RWUsBML616G`xwR^k&c{t@PDaQIIzm_-DkVA;hC5OAD<uuI}C|KW2@=Tl)R?dIy{r9_-uE@&ws#OkH3;uoLp0fumtyvSZ6pqN-eNjx)E!b^ggc3`Beu1mVpBx=c#u&1-c-yHbvlT4fK;zr*YcwkShME{V6wtz|E;fcG>W2gHzto7?-KN|zSW4cUn9d6LNU%~Jd@>@v0IIAbcXdJ_rScv~?nH;vAbuZ|*S8Bh2+2{{y~f+x^UPAA7gmphgLt=46>7`G2b3r;E4Rx1>@hD<rW!QIHYccg?gw{Kq~`_YX2wZb9KGn<no)4%Uet&8S-4adQNmX+<}JXh}17BSxgXMRTz)Ge%@+@j4HY1(FYe#?P(W!HEZPs(ngiL2y*PNeO}{DVK{C-XGfa4Apne*T_tkM!f#!HxS3;7ff>XDE)Pu!?g;7Qt2&=1tUbd277ll;1(O%|ep1k0sW)x{2aLk8nr(pkk7TCs>5M*dw+^01<R~tPGx=!mOz}C^e$S5y}&UB3CcMovjBO`Bj4biXmgrsl=D*S1TPp{9&LNv8DfK+&loAEo}u@en{;+Su*#?3eu|3BLo$bIzvUdc?~n4=mn-hw%8Bll~`2QEN=)m6A^z(ZLWN}>Z!UQrT_}9OT|rBy-|7Q7lfjM|IGOq)4&wZMK~Gdx`23;1-Lk`0A3;o{`aRf-@L0xs*^Z9Kh7rK21lRJ8yABXq4p`m8M%Om<H(p1`mEf-OzPiQiTJ>>samj|<o*{^kA<RUk4gN;kAeq>>o5wZ13qkB^b7`IiFhMx|NfWlX|AOkl|y$Y|9#c+o88D#U|pq66TZ?apW}GAr@W%BoxZkERny}Esf$aDeQ)!pdS`<2e?=Gs8z;RuS;+@@jWZ@liMCEgSZjks!5F<d8IV~JrQXZt8zNS3)09CQm2UP+=hX{tR07N>koW;4+!}u38r?X3aPV)cc2__DjjxlqPqLaTO~7cO?mcEL7hFdjXRM1dkk<OoFB|xUdNh`IT7C|gjLxGt?-->1U3l)ENW(0T;g}oG^Ou$b&f1UIWufH8lzZ)#7E)sE$(_5>Q4FO#l#Z!l7gwcR{5xYv);mnA^?e7^g>@J3TOL*Ldbb$t+6<Bz1g0S0hi9|@zOr@gR&?qhJpaRm#=)3J`5Bc6F>V?2W~UyFt~TSL_?QP01t_lnT3hKxpMo2s#}FIlgxXUf6L}%rnFwj|u2ke>1-vTm+%2dCU?Zl6eP%#uNRxqHLC(lp38_XFN0s6cP>!K(3(&v(%CkSsLKovH244qDkN{E#@5*oFKV@{eVE)65Xm~M1`6G%y4H@Wq%OX<a3l)<6gm@IbLVD2#BLAxMeR6S>mNfwygrxn^%!>W?Q-S~16mKrOM~6+5JU8G;U(uCHyBqHR?6}E?;YNQq=g`1&x}XNq%!5|iPe0tTMLo>utY0GVmk9<jWsyhCm3(A+v}~>-(Nz8AGq_8Y!frHvC9)+R3zJf2VI91n3IGqpD;)9|Z}V5V7e8Q#tEx+{KFjrwF#mHIGVNCOVQM;(igZ#@DZ~pZm_gw8TWoN$)jSGe0Pm3WiRQ_|OcmA;V(`txkP>=SV$&84;2Lj~E4PIQ_$K<*i0KAv08+lY*-gGvX|a<OTRnp719+s4pQ`-3)OKGNSY*aVB53h%{&RqNI06c@FFaR)_vz!>VDm~-MAk`11&FriFH#5N4tPfua6$&*s&A}(?7Wki<if47c(VV#@?IHI7-Y16JNNpHXZ!8`19H%&eEw5JnshSf<TO}`0|Z=5e)DkEE9{|jgY580g`lI<k~T+!lnHdZsU1K6NBZu!h!PFh6n!{PK}>^A2_COP;XUM|L|h2@{;>UxeOiPyfHJv~hU(;&wltj_t&{|8gZD=0HgYELm9725?rq@8+D5>nE5^YLJ!Bn8hq2uwBKO*ya0F*MaI?8Ohv;$|zLtby9pDaTH+s^(xoVh~cso4HoigO1hwg@`uEHd#<KA*2iEavec!{(-@`;qjD#_u<GyS+B&To;^MQ6!fa3QQ2wg@%Fv7|`&5m9x@YGbn&zMm5$6@g83TcloWrYH}GJ1#(q#Yt}qg61;FmzJ3JN*w?p2m#VMi{aWYg$@ykFVHeBdE$wZbJcOYdC*tAf1Z4%5XO1_KNs&dL5#iv)l##5%u_9+d__IjHIzf;u5%gEa~-$wAr4gCc!8|5)k@_ME;wFFVpX1uHAel_pkZvF2cM{n+f>7SoyIU)AxdOnzW~&IMYJ`-D?`>E<t~(Ub=J+Vv+ZxKPOy7*1VSf7&0IK2J8sxU4Z%bgg8nwE3NOH7+xqJ|GYBX*0U_BhppZ}8u~4pORrZhZw2ZF{b4-&Pq)TU>rMNLGItLJBzvo5tAJ>lBGM8Ucq0PVD34<;cjK+WkDKKw5nd{}q6d&u}JN5#uW^7$+?R4;wC7nwSP_545L$r43seboWplpHs2p9((MtWHqks?)~WmkoRvAHrn<Bs)HS&#W{l%(+dB}s5K>0L4X-4k0f4EYv7KV5b<zE2DeYbmPGDzN1$v)UY)@QqPoDNc@6?_&rpW<K~nlYJ$jm++%bWxgZbzD@KR@MEti+FKrIx%K!>-^59F_;<(1LzF+ve5xn_&GFqVv3E+|X9egI?T-_b03gk0)<6Xzz*|Mt(tJ_)cPsDmK6d7e@WSBr4M<*aUXn4EYBd<pYBW{A3g)Rs^@5SuH;WsRKL$TBpFDOzbx^YJBO=?;^yC(wEQ(Jq6d;|<^_0>cjHMZ+nq%Y$oIc$e(nX;H>zR`kl)JPVD3?`+aUKx)$_QXEflO+;1r$exZ_m|Cu!0m3i9eiwUy1y4MHrF7J$J90(O4W8+N2fSSLy$;xN@9%a&-wq3U-aFcvI!4tZa%U07w=>9VsZ;Lp+j%`5s%uX6l+K6{t8;OKG4?%&hS4NLfqRSaR$Uzm3B5$#ZS8=8j|d%GwWnwSAOS!QIz8zh^}sJSotN;oK!HxC`x8v;<D+%{0`94Z^<UdWltyV#VYAd#cQpM9L+vSX7LL50XjcCdT_TrGQhEQ!U}^1l?IDoHhFbUCd_Z$WOrEU%W}6X$!5ttoG$T<;R8CAyKG@HdZrHEdCbU>@Yy}%J^Ii(7UlJxdt0JbLRHN6lb%|dRKDmXfuP8$;Z45qS@Cqb}xrGjkh0f5D&&NQt@^KcCoBCG>Xw?3mKsXwo9A|#<-+ak)Qdm4tn&pLZ#C@C$X5kfmy;V{aWzYk^`}8jCuG?x+|#081auQ)vJ8MiPU&$_>xV1c}`CP>CMpKoYVHn^k0#GqkJ->n8-Fb&r@wvMLVp_=^g|8uCt4E4rKP0&#*NrxHg=-2dU5##N^2e1md{wiOq+WOirS>P=-p4gl_eDDyr6Vky{=@YwxOZsdW{+jPgPnK)~PjL=>aqB|VW0PxXvFPysn30jGZRZypgKNC)xDQ1;()b6H>tJ|5|>oLhL&BiehXIs>F$GD1ql$Jb%oJnI*XFW3DB2I*8flUiyh4)?X?_9CS!q0UGwZ9+TgQOGIT{;=RLjxv+uy*VZ=7}o5`OXdsIc$qP3z5Lu+Yv1M%W!&OA#{ZCY*J95C7gx`7`gSJgt}k!QL8-X8TN&F^H7s{bqxwvHlK+*$2aNJH_^Rv{d$M{iM-XrYZnD!K@}2L!N&|V>!dJ(DSLN$q$%xj0Ggm&bAcIn9Lug5aKJg5f6-ytArC6TUD-pe&#5?ZEFoBVK(W%MbJsXv=7sM3xsIjP6bX~`LoyYI)<Yq2LFyvSZ#2Jb~oHO<q8H|g;(sWyEwc5(HPi7Iq@$Z(y#o%y?wsRA^6l4d^?OCVs*3dCe$Y!){4l0TyI`CpE@<q<NS(ReND$#`@eY6%TeUXj#b~4AtVzQb7+1%bTlng6<9&RNL@D^+{2cD#+V6LUd=H)p}Ws(8VU0AgtO+ak4w{j()UEXY{qgKEt-Qk-Nev$dg6`64<ZOdi)x`n=6YMFJm<{nnnHNsJV9${~;;&Osx{GW3I+9||=B8G@Y9R%5@QSB8(N#wTB;SN{>QlkR^)={&&0`E50_zJ&4EHwt!V3H;UTb(rB&vAfor2-YtwiQJHLwk%1C{$6F1}3zG+K_7*jv?QAj5$NmgfZZxSlt+UNxvyL*59ZjHTO&??PK+zJe#$qHH-!SbFSLrBu(R%6(J!HIJn*^1o(D$s*>l41AX6~A<Eb_57`98f-Wp?WHokI=i5XJC1T2zZl-OJ<XUiYeuacqFa~Kx;r$5TopO?YjwnVMKeL!%VQ)p7a&ekmUW{@8!8I%a>1xf9`2<XB9N%%rUE+uuQ@%tC`-cQ%cQ?L>pe}L_tLszRQ!>XXO2N@kdi)`^gPdN+ioxxDgSH&Cok92thA%hYdPpghy}yDo-`4Qqk_SlcCf(tUJKR5SK{A7_Di4)|$Ey=I*Ws*r2#!_gmy4XUkc9CLV^bF4rm75k>3?~nz>gCzqYE2T&#TrLdPgf%3{@Vgs3S{ZPPl(WfO~-f9H94RF0SKZ(PMK^Xi?^oUtac$&K=y<&0Un@i;){k2zTZ@4^2~{?PXuz^*bJK5tCZbDZ%C}&U39R<wY&V4QR%w4_rj<h$eHg(l%3Q6m%dKWBj_dR-Kde&BjZD9gj^_*x(g&+vnL@0lpxglfA-mr4SUS-;q7}?fe}hpz65sjS4|nw7EoS9ESsCXXf~LB3&D{lwPpR$OhF!a9Nn?0cBiI+`o?5_1{-3UycTCh)MlvE~Q!nH&(jx)rZl%0>Vd9!O)ZtO!>S{WksHqStnncu_1YTin(jvH#V9YbzVVMD%=d>Q>fGZBPN7Us~{Hp!nitG@>Zh#54U}AoR*Mu&6gn}uLw6tz6=WYN;ghVoA=%Oq?t1WK@a-$Uz3@Yz;b07C2i34B~rnC1W&TUbN^lOG5jq-p~!eA45TCBWw4BlCV=d3Z)5QG(uh8omVe4f=}kf?RT}*38Oc=y)hv@KIBGV(OAg(3iuwghV$jz7mY4ZJ%~k+t4ZL}nE_xf=sn;>}>gYClT3m?Z0soWg^;CIZeCO>>Go8;h3s=yFOq=o7?QdTFMk2j7N^Heo#9K&;`3eYTQ-DdfN^e}fL*uuim<Ie>6K?jDbH-Cpbt8jV<bv}6DsK(mB7h5x-D!dyQry9-IDXsfZNNADJthlH>|l;QOW7{eh>UO8u5}mI3Is<=h$fsxkyZ(`Yfo%gtf@I%2JRtaQ_|n}JV<M#M#ctrPvbP~J@`+~F&B8RUoiUSDymF<%3M5x+a{8}$~*@0NN8QvCaqZz6`6qIZSuD#2BBE?W`rm_odqh^_z!=LM=Pr@*=;F(J>W9VNTcGnc~2?4&#cW3KF|H+J3{`Peq9l36Crk@$xH<|Nfc*A$}N>Gr>}n^rbi+8DYw<z<9IKy1?<wc&Fyi#j2=GvcG*F8L7?y%r~QkX_pV}u7IX7FUWaT`wPh*~%Qlky7(3QzA(!9ZAusZ<=@~s_=-6=9wFN&?7)~JCMbWNj8#Y*DRq1dPL%Pm^o2>9llun0b+_4!Q^2%4mM@6OiyXO<;PAq$yKLdj;G?WRs$3;|?gHa+gmPgT&B=Bo{X}Yw{fB2{<t*)gouoY7fzy0x>wKkhrx5@{0w=Ficii-<Xao8D;z2vM#S`PHXoO7wdgMb$jq;-zg<7yJ(hZ>@`*O-uiilN_k5{Wr&qxTWfafUAp<U1u0ED6pu-v4iX;phYaoye|05kJes00D%-0ldx%w=h*xvBYQl0ssI200dcD",
  "wrist_camera_so101_v1.stl": "{Wp48S^xk9=GL@E0stWa761SMbT8$j;rFFsHC+HV9ZBA^Tola4^N@QTUIBx`P#?Y;tC$LTyVBO?v7to)laF_BNB7^J^tfAul9cs$=Xhm}615q&UsCnTn0><UCi$zl`CwJ8moFxpP7j;xb(Ch2>WzmqrWZEz`KMvpJ8Mw$n&`~V`bpRFt7aGX1E<5ieye4x@jWDq-weQrgo4?zb3e63?QsJA?J0^Wi}$LbN3hvC+vI4sj@bk(DluZ%9%KZ~{&>Oo(&c4Hl=N6FniQYYKHl9Dt(`~mbL>36Z0tHWS8Fz5k7AEmACJ!~_E}+s0WW=U7h=159ZGH;O?a<ub0sT-*GD4!w5BzvOcMN=Kczu9BVrL>IK5q?@})0BBOz0a_PshlvuL${@O5%yoy<K7$IEY0tuFeHOput6Wz<o6y}HPG^^Jld&*2>A_X&S2h>P2SQj}t!_AqtBnc%W~^ujh`4gCnnn}%FLY0x&VzqRN&+g^`J1YAmcN*N)U^gn+mS5hz=CSfPiqZubl&-3K>aCi^R*Dv?fW^XWmlFyX;PNzt@S+%ta1V}qMHqus|pM1z7mssX@Dnpbi3ByX<RJvL5y;&~;ynchb)EkJGdG8~iVfWM{%T_J>%3w-pW!EFKW_e@S*=_fc{YNGRV}iLF-I^UZDM|EMqty?|IvDm)=r$+b@E-xKe=r0MR7z7;MWtw3v2GZbq=HcYKqm5}tjOK4F}Ntf)rorbFM_db1ut<jP=Vpn!W=eNiO2QVgeiA)r~>({UOiz4!_$-Q$4ERtOb7dysxJOntA*3PgWFl2%yu6My6qH6-`vt~FE@`B@J^xI-+C*ny=awMKlLtDQ1rc=wd9&uD;$V!t#9B1?kT6|jh$D0Rab`dr}r8UC5s8h&+NGX&(03+NOPk7+@Y#HN=h~9k*H%hI&Mq`-ct94;lEVv&aO~q?8AzyMa((8art~c;jtC^6^q;G6wyDTmzp0${aAeQuSFwW0ztKdEZ<90f=^<Pwt+L^!&#=6I~)b9qo?u!@DR?2z)HB6VTU|QppZO6;6(bipW-xopzWd^LjO2Rkpf;|%}l|m*su<*N`dt)K7Lc332xFy&31VL<xoJnWur%9Zh--;7i%Fk+_dRcVxNOW^US)d?D`b6gshf&1*eyAfX|i^Z;tUv9=BwTO`_jZpmN%5t>Gu*)l#{3(0q#slPaB&atub?J#E5|xaOIn!-x0bQWiKK=<0;KA(OM(mSU5gi2AW)MsT=^m&R2pK)&W~=DbVgoMxfd&2HjuVPz}Ooxgx(`(9PD)>##<o;4KSUv2&9eF<}DhT?@e(}7A}Yu|D6P_7j^OB0p4rru_9N2|&NwZTX6lp5vdTGtk0-PsT>dETO$kSMJH2`poZEeqXio#57Ud>~&~F=!7&g+iOUYV8NHo0Sm(0K_iG-^V~z-{L00mP#>;_i<9!AS8}UI9qVc%N@i~h_%AVZshs>iME#tcx~XXS-1Qnlc=3C(*_^I59v>obV*w9KNSy#Y5{MMnpzw*t*qXk7z8x;@kvy^f6n$L>r7-It+E20fcFTQ*BI3VgS32=6pWtHpZ;IUHl5g_^VhAS240QJQRd4<CcCmD1NqL$==({krOZbt5TWIkCO)};U;gW}4BUECKhW(h^QZ>o3L6<T{|G$z7w;7bOQvFuGNZygucjO>KDD)dSOmX(v6l8tST&PkuaW0F0_ES7W^BH5yk)Ai3qD*<ioN55R=%j{`Dr%sP;1$Yp-u+~z8*$0iL8OF*lJ>wag=!OeV?Y`nUT32T?EZP%}pV0!NRazfga|Q?!NWgXabFluc5uTawBh-{{}=d$SaghqB`|TW*>Q$9qRYzHh2w5*><Be$)Z|Lm0zy)TLfL$+>=Hq40fnYM^FLeU%vpeomo-afXdx)Aq8LMXI4i1PO8uNRw7TxByV#WDxmNI7VFZYs`%VS1z7|Ef^OyEhm2_l!YGt?$l^Vbbj9w;62?6!j>qx<9KT6ukvIoM2Pn7)2dQF5l1xfbq7-?`jxY;B?iBIykB1fJxibZFV3#R4;h_aJFHK)tj35Lz4Kci>us20JXC2|Y4CV0e_|X9!^W1zm2?N5XCD?_xX%tb1O|2_$ILx!Jn5GBV{PCOO`BSEJv2?66xG$i#zwJkG&99$hFMFOHQETrnk^o4koiu38IOo@-+6A)GXNk&N7*(TtvT3eqhR~U=tyw;iakXX_s8|!ZR^(sSM%tx#sUYeixWTw7EA}yLv*&^{d-}s!@;yEAlon>|TjM{q3c2_^*?9KBAWIICe3%(hU5Fk47lRSkXC&|L5Mi>Kqx11X1yr!0gIlsnhnGYFKg{|m8+8mTI+e&7s|6^{<-HpWOdvkX{<3(y9u<E^>*j}~DoJ{ebM4{Y5gL*4`;B@IU~svZqGi4?V?5pRYp4Z2-rwW+o!g%B5;oz@Mk56Lguv7?`c8};Fn!0Rx_Tj1*7R~k?DUV^43Ka8#TU-P8VAhlr~5l!6J4t$3`<<o=<PZ2>VE!u2=wOtLTfby=xmj^np5?1k4N-k2{8YjASZGQv~IB}q{e_41SK*i<Gs@*O+nX1+(m;#4&|1wZ_dAoj2Y7&x4G@l3?l@EFI?pElYQPh)>+I~#X`R0ouZkZU7Q}cbZ?{{JjUUswfz+^)_a0z?8#~yftjN!;`L=qI5Q<knYkx*|EDssKvnwp7<P+v{Y6<MrD24-iUZN=4ZlRc+jIbscE3pWLQX&e)3!4CG_yte23PQ12_jTh)tEnf5KX^=gM7pTuAw1tvjCaGDfbtYr%M_NP%SdtT#J-+oF<EmRI^=eu)mRofyuj`i->8C3BXoxxG4*(H~iqU%i<eB#M3)X*4qx-h~r-w^5Kv^1lzBwLt}n9u1j#)I4~!Y5feRGBp=}P0t-BkO=eaOrTYJqonR8H>#{|pFojukY-IY(IMKJ($NcPhx)>JD>rn@R0Zd$Qikcnn`Yq6pSH}s0Wf50kyIB0EGps;FXwQEq5A`{-m5x+)P85yFti*bfg{PHj=9{sKtO9h}zNs(7us$TszY*sFOBh{1DCb^p6Ck;?ayX}q-Vl?NnoE_e(!|K_5k|{qkSZTWT;*1+HYyBl)w^lSIQxqf>O<(ZD>kcK2uNfXxy&n-p10xw6O<}G=V6)NTWoV7-2;?wF?c@#em`YGUFC6UcdHd>>i}%g>5z|UV|i0=RCOTgRam@PF7u{S2+NbpjI*gV@zv?Bg%f~8TqM2o%;L(2`E@F_U~JxT=weEQBJP52%B0mg-?+(+4D?^h+Fx4EG&esjM7Cz!jL&0h`7b1Z9AsK~M-hRXxM^d-x<Si~T_TWfj9J4^5V(*5&sH~uH{gI!3&XyVer7qR!;^_``U5g;6rqeKq~(H7-Oia-O;CXoG>l%&;;gOwQB%XL!7PeD;O_i`JaeN(u*qfTj}8O8u9ogo$M!^<uV4oLyA6l~+S$k#%U*;8z79t%q)i>F>=MChw{bWymIK{qS;AbUl<+!%&z?FR(aX^vh$*B|>KTVrzlO^qnLT){KI8P+G`L`1+_qamP&sYj9Q1R+WJN{kC42DiTZWN-U2%X?rR*o%ha<?*mJUP6cQ`D>Ae(I`|5YeIqmV$JS>qfw2>y6Cd%ElCH8b`sm*Xm>bo`;@SW03yVSmSbuxXZ2G94T5s56hRGRR~vdb~X(sH7A}@w-?_qg>zkUR^1BOqUotD(Mymtp^s`qGlcHlxAI!HddI~#re%7IZ;Q>(0V4MitC(s&J{-g1If=G3oO$qHc=j555V6dg~U}-TE~Rl8SPAJH{-Q2Ic$l3@jNyYeC)!#wiLzGaGEDz5fbaVWpEzUeB8@o5iVz~2!QvIqf&UUnx^IeV`VMSKdRUlQ5+ZBgZS<%+Wcd`%KXl;S3sydQO~`TAyre%N7`|6y-OQ}$ar+LvK+7G+h7hfEPyqJindg#I***a2Ts4BKh=DllExFaf}+#%$&e9r+;NTZv{~Y`hB?=PCm5nV_U^hO*-Z2#_VD{t7E|KJ6xS8GvvAxXlQrBkj*Th3EyPG+mX=}c)&qmOk(AhRJ3r=ttjo(t&59<LiMd@ZDv0Q*yt1<CGLl-<9f<rxj^;R3iY%p_weB65_oRETPjsI3Go(K`F9Y>KddPfZQE`M9tg)#s@-1;=Iy@xDH!7DVBkVH%S-_b_k@@+!$dIAi>|&kM4pkYB_iZd_%xPr5uS_FD?*$XB{M(aSH&C7FfXv%E+vxvo)izcx7Vkz7D-a~O(MgSTjJLhTFJxm}s_Vt)b4O*5`p%4V<C(+;HqLx}1E827O~fsur0arH((_mQCTbdZOUR9!j64ngSvExq)%aqzPiYsz6%`zHmMOReI|}|=js;i@{Pciuule(Zg7v$l+1G3M7dTA!`-(lV1&fDq`1S-8A9#imB#6Hg5Kn}?r6v~i^of^{yR5Cr(StIMhm=BpjgW+8z`(UnE~n@Or6a%IE5qWDRMM*{Qo4bSe)=l82EFo%Zhr~0*dWwE%*;WexW>GSf`NoX9SpvxBO-bLx<wyl{!4~emX+YRBC*A~oXgCmKSW|S-Uf=XAwj19iQ8dAIMxx&)k|@b^112Gdg*@@ja&1>8*qEZ*b_3KjU<X0jJL`ZGgmj!R{y+lSS1ip5_)A=&PA@TFCt?4AGiGAc%Ah*d@xbNq~W}sps@3|ZIb`ImA7*z7WyC`PB!NW7?P%)0BezjCUur9+OcgNGGAlN$JK&|SxeyNEh?y6HvFNg+AVoal7U#HK1CpXUP+YFQwL>@w24R6?NLbwLBqj`4wp&!2cJZ7<#`j`85v@{W!>J_v{i{tQmY!vlQE3oUWdK@ni3YRs~u)a!-UDot&dq`?uS?tzCuK~`WkORBeS!+mo&$JKfB*b6}ZR%fnD~qG&+*e$h(!209=cqDqa_2y1O-sMVXOxJTZMI9%ZY8Yu}AL<XK4o#0$1;7kLtmFhiL1z1yyEB3Y_RM(YC6mK~U3RcxEB!AWr#0M$f8EdzMnNazg)5O(wfULlsB8NZE&#}rcp=aSPp?3DyPV_6V2mr9E+ok;Vs%K_e@=&DF-EU<R40+s!Q5cw$xO+PyZYFg>9wefDChn#>v&9GkFhrLmJSuq+gN2N}u;PzF_;r)TC5o&++<T1xgzMD=E#;I6deU2tNAfCY$%2<_Ax--+cNWw{mx&2r2wGyi@w*>Byiy?Em(9(=98#55Qm>(zX!O{lWe41fT-BnU3<5~ae=q|5SaBrYHlNqZegn6=@pAXR;QDso^?_F?Y-3?+gt?LqKp7t{MBXCZop@i`=d1VAMF~SG~u;ODtF_^V4Ah^mAqZwt8kfIPE1R-9j29!C7f-*8y8>P8O6T_R)2>4;BpUomIR+5SNk+OJG-C3&vC~{ewPGzy^O0?;gpetsDwjEv--TJgID(EuJ!}AeX*Q~joL)SP(z8JI1s<r5DbqqTk%bV&<o-Mm%@U3?^U-VD3Y0PtG#jT5pw^%8wKK}$$bweln%O=!%m__X8wYX!W?B%jO_b}OU#NA1JZos!)MUU>fhe3w8ohIN9A;XsAs)A7-5dU}>Zy24UA9qTkmZu1FoexfEZkvi(?45I^U<)L17P||>lFOW0Cp0zmyAYoeS%Tj3y#M$dsHq3h<fv&jDF#4ZKX@a1stIlBDmlv4N-P2GKoW0a14Gix5v$)HFtT4|*Yd*{Ay#@MsJx`di`2bKXY4+l$3iKG7jT9C8pf_J3W9wDl9^?65|-_&U15#+Y>)#4>l$hOjl#&9jytceyJwm!JVxf%9BAExE`&KV4o78%Lb-?MlN;?Eu)7FAr}gO4E%u2z35vRrNCJV5#8<HsAJn5m9hdwKk0OqI$O}n@0~8)lK2KmguPGyOUdc;;tH6AY{@{p^GWI1AlW2dRmwCpP3l{)aLw9Fs&lby#h^5OpHcDEgFx&ak#~`;e%)3YgZvCDn*dcj>@{jSL4Vd4EV|ZO1HWxA@@?+=s55(IpxmjB&U;Fd)Am!Oc6rZR)kWjgdTaRAB-CQ)>8<bUV2#cp0G#Y22+dQqvDvL1iu&^vN7olv%UV(d2{+KC~x+Fmxqg~pUxNmvc1Wi{ZyE3{Fd8H?4<W8*;x#1S#V@<l%qO_e!%fnp{;j4__9G?>5Ve5i38te62zAhYSF8#=ol=>0wuj@m64o+PM-e7B)@_{GFF`%1m%)_zxcufPsQtIl8#t^k|CXpb7fux)yP{}te!#Z}c#T~}`^jS!6X?JV{wlF~HrsU8bTM?Ntq|(+LFDSn3L3~t)saRDTN_uPD!aiK)n5vooe?=C3(Ls6^8E(MC(|{Gq!`oJj;?$I31JBq7T>2mRwy(Pwah{XU1Wiq4HHaqfvDTHV?nSWtzs?+1^#6(ucy)YnY$>6%Qskr{VYEX%0W#?vh`FsS&S+y%g(;`qg1~I;?-^h(Q+SV&Z9WcgNn#wWH-L$pWt8V!z#Pv08F7o22I1u*+;H;7)~<UKU7)cZugovpMEoaMh>Ne-&IOw$1W@JDB-dG=4oH~xYP4$heMSt0S6s@F^C}0SLL7Ws9G4OHL3Vwvr-!Eb`ZUCHFbU(opkNb?&yZ^58tIA4g1?Oc#p<aukRT-^OQ@#OjK#-220j0kPAPNN#+xtf1`aQ+DvaR!^lM-gE7w|;*vQ_(6N_5Y1EY)}i%);IFx~YM_CNitne{Rh#E>rT{y<Eyc!Y`k=~3ZJ;kHlf=0)G}?`X`x3$2;j8L?J|g~um0OM0-V42=PBbQCcA7*cte8HY=Ha@)A<S}f2+FINB2T8WCO;QKmQFOo4H=!$2BhY603v^aOKB1y6}TZgj4e@g*Z%%s0Qfna!8-Z~P~%j99sajU4W`0KR({6CD>#!3&b??A7paPK(fxU$InJ_cQuSVAS3@AHifWsptUcuM8j3^pqv-N-Z+#$QI$RW(Re<R{&`p6xo#`J-E4V2XtYo3ppo?$X1)#_v_sx6PO#@a9>DhdoZU=ZcU&&5YI)h)JfQTFn@@>&0q-<JBkQ+?defvv-jp41@t>-x8|oxcsr)h%E1+rs3L{i{_ppM!8Ckut#ohtC<tSCr-VdOJNQTu}8WTgG_p7rt}V%HD+bL=#tl!#BHh7I6o)OEqMgz9}5<EOAZ6E5!LLC3-RXsH=MM-kgwF0)0?poHvHQnACb9s8rx>2xB<uawv2~zA{$=q<C(l*9{a`3P4YKq-~gRQRGc3!H69C6&V%9;_~j5HLonRLNwxF=+Q)pLn+p`Q$9Vo2407RVDySo3cM>&0Dad1NzihL*{#(41Mi({cr)48ikAohF3rvUz0fdwBXp6tvr7SS1t6PtQ6<OxCi`KQKxkYmSMk{NEPR+agHyybRhyf`Ie$`81AJC*g$oT_UPra)~jRzd^!#(WLM)G)IL?-plO<#=5#yB)0<T$r|jk2zhM8<)o%`kWsOLaII_4ecKv0m*oaYfE8$P$iisR}W!;5Lxb9A`AG`St;U$A^wGi-zFf6;C%|wML`>rHs|8G=BdXPS{VLD+VD$`9*qlfFIG_^6m^UQRNOK`&1dbNj1-drJgcPAfW7%=G&3#b`C;0_1o{GqT*@2XWPB6N_?v6<&A_ot{>Ps4~?ZLhn9T)V<7i(KxR%7Ljni(eY)3FF`v9@DR6{AeLTn%?UEF44Nq*smT#8eN*F%-*P~q0Be1`_cYp#!g~q*%f{mkW^xfq444HCrozc5r1lPrl@=HR=w>qh=@A6kxEEW#T^!nAifg(ZJ-TUU|Plcq{kQni1!7HdnasX8}-W93wT}j1lNxyao23+SZY20vfVQ}JJZHxpAc%s3ZduLtyEV=8n<U4*8skY(fQpBuF8Mf_U@%#VXDo~JgwXrxh{QDvTXeR?TNu%#n^m_w@;^LiB@~*XSW9K?PTvDsBF)1!-S%gkk!EhuC!xJEm5>x|{UsM?J>`a#hxN{zKOr->eRzBX6ro@~N)A_L%G`I55Z@m7ap37OBk)RB`Op=4L<mBk1{lXjsk?-{&Gs68w98AXunudG;a~pKA?veqivVaMPbAb6%vA}p;xbJqs*}(@0q2E^AFMq-c7B8Wp8A}Wn{72ge*%+@@Z<Ypr=W8K~CFv7){dz)wajLPNPj4r*w>4rg)OaevpGF$xlhw#NvkwrZ!dg~QwqsG*er1*8>>H(Tn@V#OU+=AwRq_lyj*1scEIA{EzdTY#R|<MBqi9S}2u2(is)a}<4507whdh_Ie{cW_hcxtnc-h?DdKpi=bl!72iaGv0vI|_PG3iG{jnE|GifxM#or0`gdnn36G&AJ<Tk(o?GwJU#7A~04!xG%X_bFxlB)|lI;MT-v3-j2DxP!ZAN*6mxB0w9d^BS|2;vJcau~mKdOu7l^uIhqc>Ww}&KMqBW=_m<<J9YF`u$Su2IejyRZA(zfTWtH9Y-;h7Z8p8YowgLv7E9BrhZG}Vz|0=Y$o*aS;Dj3aX;($@(QTiXcB9e597l*-%)vLtdkw_g^v@D;&)xTEq|~K>qUkxK6k}%9x^W4v6|V|9h^M8vNZ>z(tZ}8L%aU^Vt8%KJU8f7cDLO0oDaHz5BFWkzLBApIp8jYsO~+~=@2}n|D*=BC0Uu&KSn{7dSSDbONHn+?rF@h(2rPWDdeWZd_&e37<!iDI5YRw09uN(2ILE$1=q9OpxE8A*5K$Y`KH1(x)IM>9=k#p@@V__>uz=JP8Hr74ECQdGW&Sd~+uSTgb~^^QrQK)Zmoa^H6Y(0>`(FQU2|yD0Z<=0m9=8R9I4=luV0!3T=E;2W??rDM=SIMkH&2CyY5rV|m{Q*%a^>F)<ibKIAc+vAQfMO|SA9jJE$?fx8qPFH2qFL|i5%Q|iDIn3k|pWpE=bCXb<$aU-78ks)-(-?*DPDC&bpGh_MY#ETLmW>XklgmoTuq>N;yVT8=^@&P4Gy3AXeCGM&l^43qTTp5ccv?Yl<En3x~TxGq}RQH`60HuHw+YinJMaT>?SdLx_tUK2Ow=4lrjFP1K@P4K60-E0~wm2<E~jB+ja3_u%)Mk-4EdFU3yC_0S&GZmUZj`T!$$P;v!22VkqB-Clt_I;;-1z{pKx(C{^-0DeV*BO%X76i@P##!Bjb$mHnj**TSO6NuT{LdNeEV{Ml2zhw+QGjDSD_VgHvUw{GETXFou`?$?%DT1R@sF6QQWYF|D-Y`Wy(rrAg=qKhX)}vGQ$vOY85C}5+C$TRYv2}m!I(6amWR8e$GB$L=%{j@7B=H_yxpj<(fW@;KbFR)4l3!mSimNqEY{`HeUis+J!8z6679}ELls9s7&&o|Y5%M}|LyTjhMOhpCd~!>-TbF&;E;v88Is?K5N}Plnkp<DE8ho5ztMs(|%Sm`b5`Z}^B6z`vX6`<6ADD*mqsY#UVRte4Bl71M3N1CC2rDo9t~t|(waawBN1WHa(I^t8Nytm#))M}gw;qW%=F)HIxV496%t<^hdz}g03{wr^1O*}g29DMBZSXN)mmcn*FmumH(*Wqz2!bOm!+(D|^sKLc^<2_&R;=!{=$bP}FU+@E@_BmQ|A%Bf?q8<JhV?&%b&>vLK*YX~2Gosf-BspBq<%0WwSwUHpq=x${DTS(AP=$|HpaDQ5vZo9%FUIKzDkO1)wb&Za8~A~7=%`Eiv#uwUuj&@ow+|_nNHv-Ce@-+dW9&}B!ivZm{3yh?LIb_j$qr28kq~93f2qqwbHWchAr<rQJ;m{WMM2sooo_{`~)0c21Gm6&}T&+V+5e;k{R0Q0iVBc1ZW2TBdh%rz7-k{UP6i9r-CSDF1iz6V7MQ+RV;ybyDYgfx7<qspW}YK;jQu$qHDzXz?aLV!6+lcf~HZH5gn?sxcEYuxk=3)rHNfJpV6Uj7w)n7k#MXgUuABzdE1oimp`oFx5=<;;>-$Ss&DkOI|fS`7Trk0-P!`%onhX&{Zxyo6y!&A0bQKPC9L6J4o=%?`GcLnO<Qa6rf1>Q91RK+fgu0pGHF|eeZ-cxMVlE2K5_$VKzROzTXcUY$GNYPbJlw=D+>^$d9qA%YUTffCrj0Rii-1<70`%1pdCniqWK-9=a=@a>S+Cgi}FkS`SQFJ(59DUq&~bi7q1Jw1;E3^!R}28@jkZ_Bu2DQ*?@<eXNkAw4{m2YHp_+<&Bz`#RQpYKN-qURjGg1=8(tp$w&WPhKTz$=hEET0G!(zQ@OOs<ockfF9`Y-Ty3J1{!>O!e>~^gaaoRldr~#}5b*1qm{*k7+%Dl*TOLGe9D#wUktpbP-d!<4(O@}@UAY(7~<*v6nyDu-vOTkW;ubDku?CGjSM+1H}d-*0qbabJodj3lKswN#-fLWoz#tVG^3SXgKAmr;CKCG9ZjeXb*uvMG95AWIn+u@bOxYR=0KVBTr(&94OiaQ-`i^XDjZg>Z*OmD%fEgAOQ@KtLDhN91Wq11Ov-_^leYDnQ&*4Cp1I_BgJS$5>fR^zQoYBQvOs~%G&WtY*lyC{s%NFf#iqLbOB+qvy#LF(ZBSM?aP?;U9#O!tmWmVYvyBxA_U3z+I!aXhC$nxf-+2qnhpk{V1rQ22lv3dk`jHjdzpd}G>b)3dEK3ml^dLJE_?L(uLtv4q_M;E_|mqGhZY0i4nizFGU}bMbg%icXVOCws{IX{t&->S(|eSqCr#cv+Ck*;I_g(%C_e*kipbIT5~*Bt$#0>5aW?MIB<tEQw6G_z<4waERr-Eka0<+=McvX6g86iIWl$7NwocgnIkQBgbEE5?JIJ9jb~zfxBeSgD*a!A`7Nxi}{!OeF-NUrQ){e?YWTTv&iemqh*Vq?owElV{f3kp>-Ke#-fXLGTN{0ODNts>KfW-cAB|ox@5-<9z~UZU$IaL5w*j;v-$?0BU7scBs$e7ut5vKu*QZLQi2Et4GVu1idFkIki`AidJQwi!Ytdm;B8NkBq&X)q4j`h%M!~Q!tBH}WHWk<75sX?TfZ?T1I-#f*6jL8riH$gwl1A7R)4`yLpG;^W3Ko{l`^b=1>YjWvt<DIJc3G3E`IPw2|9z4`5o2OMXSq7H7kZk%boXYsWzu;{J6W`w;KcRCl7x5!-Z?TGVBMLKx|rZM?}@0z3?AqVXN^N3u&8m16+zq%$NG9lK|bofv&R_QNAzlAAKv{4sqo6s<SwwEw5nX2sDIXKr3Ke2)v_wATAeBio?Kpd2~+5qZfUa1-!;_lE`!9JT-iqRqfH|#<6dR#5&;@ED8X(XGQzdTiPQ|fGqX|#Bz0=o`9__yXJCVR8KtQ`0XTp-bEcJ6Afkp#=aVflI4+FNT0&zOskHJUNOt1!ADi*B53Gzd{uW~Wwl~rfD>(67pe5;J!_55JVY^K@&<k|_v33m|A2f!H5Z&V<q}6T@5r&`A&589=ZSq5VqK5x7HZAo8HJ5j8(qm|pE%53^f79iGSMbYB9kOwK|1%@(>C*iMH|{~vP@0{-^QT}+G$$+M?qeDZcBTz{?U7B>&`%|N;ki$M4LWcnn^dOWBBbN4l{JW6Ct&N0&dvT$9Lyifup<leu5?FfWF24H_kV9)gL49JMxl^5kX(@XO$kDl$$O}sqL&kcA*j^?)AK*Er&6S0CEFQg|O@?tt>!Vc6=Rb#B^?>qr9FfMRH!CCpSBuxim5st%Dnx%Tx{;3=6yKqn&9VK8>alG=`d=ua=YY%>LyIrO36jD1^6A={FE^SQ|j}skZ~)G3icgcv4CtyfdFMMg&hXHk-gy6DkgY2Tm&YZs3vKoXckU+%8b!b=zQy4XfFG*OoJ?Mu1PXE1zp)0A*Zzspd1QSq`37dWe7B1`!wKQF<>v*FAj*Ynhxs#F&p0MB*<lRerz$=R$jo{YV}Ka$W8Z6oqh}va3Jq6@x<>X>!(cq0zc>LO&ZXM*Yf#`#rQidXZlpATFt7{#M(1^x~}9KWoSEMXvLwn(^Pmk?{vXm3g(bIq>e6V(C8pvAovZ5QC4j`VB=AZCE}5rH9Rs8vfp=XvlnhJ!7K?vc<{>ew?>p3X;uQTnPNa3N$#X>d|J(UlEirkRO>ZjEgoIH^*TIwG#6y18?>Gah%c~6lZK1?|um>3pjQ`>K}bg@@@`&_sFFSPYRM1;sNVAS59GjH^uME;@lBl%&b*f1TPNL@`faPYj_%C*;jjQd@KpBR`G{tYNdT&3yI1i{TEarwI{$V{XAg);v@X1cl}W@k)Ry*(SyEAAzBi&agdGEn(rUZ2cTJ)S`4s?OFS{mT5a=e8?$sfj4LF13il7g``3p*s1#vQ*siOuQp4|*Y}R2NPDTm*Ybi523<;QbM?TcR%mUO#K?o|D>P9={kh`A@7y}e7V2<L^FFDc5PbB-*;6QIYssVr7+$_7-jC+|Dj*x+>;o&yYD$iI4$mm(7eGHj?U@UH0g9Cm6rrD&VAxx74b|WC-H~!E)QHp9B)#FBLTBE_gRSsj8;;<VdkGMNF-ME3P;nt>I#@A4RoNHvf_nd(m#3N`;%8U3@-C9N2=|PSi#N7_#zY{S!keRGew^+daDMFf4Vmp*Mv4@s+^oFjzOhbr*MA>$!R7G`@1ytSdYe5r8E-mea`-y4x^kol&(8-{nyKsFm%3^AlL*e;@J^-%vp>}8*jN`@<r2@Y9(R-!!uUh5h{X;~H5rZy7;Pnd^7{m-g$2w*MIO0c4Ho+4F^Z88|=Yg_g04c~krJsG8zd0`8MsL8f;A`eidMyvPxrtdIvkyA}5T@9TZhy+X_**MipQaiA>7Db9$I|~JLYaK2z-j7!Iqq*vsw&6jTb7-z;|vPeOWj3E($EGNxnN^L2{YgjqxB*Ji2=Bdwx-Qqf!{~j`=<Ephkm;oQ_s%wH8`rhPxITE)m8;O%Orx&lh1nSY%vMnr2j*JoW8G-P{QALNj(_};DELYZlMKT(74iF4F)MbfAMdGoJiH=w=dlmezqxYVQ)E1;Pi-db#u=f<cu{&S3kmof>-18_ojCyCXvZ5tR!LAi4&cL5q*`)%gG^Q%!R|%rAHHJAD8W4(jd_Lzf#0De<wc4ZZWN{Qw$kAT_S^kY`Ns5sd>t$gLL$VF?G=30F5A_raSeHb<hZBD-*@|!Akt{j%;^)%)T}tk_X^~jH-v~u4dq)opCLAQb+bYr@7ZXg1*9KV6R;FvGpCS5`ld>?Dsza8qw3rp*yI`e5&Qk44a>VVAOVP*Fz3F&Ig%GYq?k5<<u5Mk}pyChskD8Ux0q$@iM3ql-3X)IZ!0kan*aTMc5U?6XDZ5$hv@S2Q>?!PP;lbru6yUwBelD-)WJsTH-|QIPqkr1W9Jb0O#%!o!S_DuakVyCTJd?GriY?ctR7WkLu><R8m{}$fN2!xTbilCKWNv8-e@?q;*ER=yV*gFguUDL@A9=O$h36Z3l)jO+S&0*KTOuXY!)=vj1bHOvZ4j4ts1C8LDYfLcn*+Q9N68W6o?geI^8RZD9=Xd3Y6TDUN98^ABD}IyvW6dXa`s#ks0kMS|F`H)N8ev6fCdDp}F_MG+x}5!O8!FUPQ7>qysW@Qa=epw?zisT;W)&k<;)%{#D#Bf{<Tl3TsJuOZU}M4F6W5<9tNFg&+c#Ebry^bijV&v7Rw{OmR=R~@+dyB8$|@SMKUy@t|tNxdwHJ&DSC<>iNan)>aG<C)HF7GJb;s7KZmT%~3PZ~I+Z4|d$=dfRUev0iSyO5+7LwObf{S{9gp2!Jd9Y^Cz47wo7lsN}wJES`iUw~eO%U5cFopD{eVD&n1<pjLG~>S$xacLq&U5m(0vVF$B8G*9&QwLF4o@jSI%Tikhp_l0@rdh9D;A`sVIl4!Pi*xEBw)a!0<KB~#PPVSPCX85bu;RASe_|Tr&`A7n)vRGLB$kTgJmyyt!Z(oh4H8PFY*#EZLo1MH7GUO)U$tN)PWoM;A_C3?{k?KFX>=7Rs^!DYsh#1upU<XM9`xW^MqzE16Ip=sW?z^~_1bJ(4X2Gf2^yfHc)zh6kwMWfmTjU_IFp=CeD&%Q;gE=1HAnY#k=ICN(KewBB`~S_GP@L#JrSf*ziEX^nFaMe+1|SA-#&PSwujzM+bt_t$`SGI$WeBmJ`-+BvOE(YA@JaR+f3M^I(`k;Br&B`GT?NHo0!hd5U0PsE!X(sT(+C+djMv%Adie0%-8$L!Zh$3TFR*qFd-eFS7!J63m0D~b;!#qfe>9cYdbw9SbjGg2M}b1F8_YH=ymu6pf!Rb1)OnwrK#-OSI8oM;a;Ln;TWSBE3vpYShQ%$U328^U&4Gi~OiCIZL{y_aXbV07qslPsEH=E(_xFC|WZmX4cUW<9@rTwM1Zlh2@jqlmXo+;U_D7E}4}!ATgPa(Yq&GI;PFL8X%QC_)6>^-Hnz`S^2z<_LqYj#mj46tD`|?VLCHtK=eVJ%P1axLt=g;Mg#v$;%k@(@(i5FED1}{0SHc3~fNTLobrkr8W<<~HP4HF)+O|=Y@5%Igb7^_zSBy^Ii)G1H|dAP8rokvjC{Zi<C&eIkjL%i0*&((hop3#xsUUeDI9H=*O&fl_+No3Iqdd%gwTy>1b%c}X1oH!;T4XW+N2Os1iSz;c2-naZf{aTEUX_)#?(j>)4pzWix-k2zGOH-EKYRqw|i7Ab9jE3AOU+q(#Bob1CG>a=xnA-D1+L7q_m4N5G;FVtI`JoTCtavf?@v=+zDpW#*7^a-C*x3&>@QT8-6=j97>jfaOGBeyt*5SQy+;;<Y??9cd=ERonU|J`jp9O3pMv%WOBpyqHsv>%O1YNIC1*bA)QGUesSYzF4su>vg*Q+D90|b40h=P{!E%8LzZHo=;vEC>~QPQY|Qe1jHt?mhh80-}3a<E7lYEd#(eV6eujUWCv?{*_`;abH2s=x~K@2{SIh~*~++qL*PAfQ$01R0E|$<qk0KGE0z26K(|5NmeV$cThxlL(@`)QRwtO2&T92o)FP;$jpJ4>s=>_#54_BEe1fpJwJQYt~i$ko{Dm;J`KjfN_)GdSLCFN*t}5T)Jamj=Ib_`;U{pPCDOGAWz7)-I8=a%_r|lCBhhVhAL(Jz3EpjUeB?XueWti;Ti!|s~lyjZu(1)5UM~4fVGe1xJPKK6t^>)5#nOCPsZuQJPRZNwP_z2<5{{?jRy%e!WeeX?6J(ABds6p#Mv}EU_ev<hZjQkNp&giGmCWAP<Qn8g*v6zIo_Iu7w+Q<&hs7k(C<=~>Cp^^6$@QQ9#+pN2kD|9fWUM-#aPxXnX5bft2OXr3tnw9@~{v$N3W7Oh5{|T^MD}D3F?@;NLk}{@a8UqLGY=9X8+Ck8V;I;XlFz-JP*>6<OGCU{3-CSDhd!nb8+~W877$gI9iS|0J2d3bcI&(R`l3n1%rDZh5aQ0aK@Oq5a{W*bGp+w@q?ib9EW*QcHC>3Iu<e!_f==#U{nW<txQb#MYKy?a^US9qI;Th=kDKBW6fok6Y*!ys%cl?X%NM%6p{wnRpuamE9Y6ir-4_Qq#Wl8lp9pT_1H>$0h@evfE&m5WL%5EMnyFvmN!uz$c>C2M6&nXceN5KHx{x|MpPjamR|Dg;xX><HGu1o5QT<c$5k1r5ays#r4Z@dgT>Jg`lTKVM9x<HN(kw2!2?MN`IlCv$cj~18(wKk4`jLzgj<*V*X+-~(Dq(GC?HmeIs-n{j4yabtNB+XSlKlex`#BLTCkWK6~K%3<kBhfnLnmuV@>iDZK}FfrBXMaYqZ69;NnI|*gkcV-s5fy5|bkq2%-~s+5GX?_DdG3=uj-!DSu61K{s~-fd7zEilvEa=STB>UwdYe^HBp*(S6Qf1QKx36C#!0ejqP2u;kp=Dq8mJc=y_*lo%%(3U4hDlwlOF#lu_qE2*UnD`%_%#oF}iZO$ZN>vW~mFb<3)4jEo7M846{oC0<Umn=|?tb%3vfF*uRuD6ObZ&%1W^p!>ZV-Am*ZuZH>gq%-9(cjUhj77fi4!F0B+)T`OEk)vcFwZM}T1LmoG_y3YuS$`JGX1ahZRr6r?w0cgdy|?=#>`}u#|M2W#Z@w=@RK`%3SReN?#Yk!K&1D?)IGNt>V;g|vu|cQWv`<-UC>L3FAPF8t(wwqux7WU78;Nn0h}C@7Km$i+J_=_f$Z)aUeUx788)SJ*sBJqg%os4sVk8U(qBfAVN^mTFf{P5c_;}-Ci;P58Y4wp;&8NjYO~`S2dnQL?Q}T3!Q(>_`PCI$!9#bP^Ug?UNc5SRQdP$z0DY(GuXU#vm~jhWjrb{ye3t!bIlVo@J5}+w*)%=8I&2EGQXr4)Vy<-`KO#;$=YZr#EzA)y+0oKH;<8?Xl0D)o%4SZwEH#EBQqWUE&C9{}#L`I_JsYN9Vj$h4Mwq&E@!hGe^&-MObV=B6N2AvYFcmn(9EE(8Dmj0ou+)f4PFrRrzbZ_twTYTu=_aHmn4hMzhM<Ph`|%MCQfXRY@VMcUA5~xIidu%c*M1@$JwJNSeC{mZ8-Wjv9h@s>l>#Y$?o8I0>k|E^(`Am7;Ja~BF_?QO(|sW$4dQfSt-C`Zg)lh`4uu>!+DI!TXP%_R>rM<RDO(nsQX(Syc;?fx+SbW3tpvyJf#&FEQJ2?w=|S<%PA!xF6{>!-?hLN%AaN%5uYv_+p7SaJfD@z<2=q9!^4)Q=)Re3*)ib95HcmX}>80hK>5_u>{JS;jMMl-NRF3GQG+X4?M#b>9r(=HL;TjMN$$<|_@ucxV@`NX^9^=qd4cIYF_*s6_hnlFKU&}$w@Q>zlv6WnYP%AQdro99NBlVQ;g41sc_O)9)K+Z=J7;z69NpbWZ_Xw<mBogb})!0$#F96blXS_3}WUD59$<XSW<P!It;OCc7&zRj0rU=UQanEJatx2>`$oNJjYP{JD9o8NUuN$s#b`Ay{fN2_>ls~R62Uyx25Wc1OVrR51h~Ii5mxdIlZgi2@TFUPiC@oCkR&i#=PicOCr{k+LfiQtf?u6@iEOCb~UfAAZdSX}u)R45+uE%4!tl$e%zb28d9FpP}lPdNhu&&vQSshryFtL3!NY@`g-6JO6ZiZ3`v##|_))ucAh6~*tAIjWc#BG$!)V2=`H`mRd-SlubslhDQHvK_m)?K`1xZdRI+xp9)sCwF+d;amIUnUWGG+bj8n0b-i&hPHDUTpc+vd_Kl_mB3WCeprx6<K<0*Fj~bS#f@mp4TjHyR!!qvyZlO37-&5-|JhDI--qYEi`wy;l!a4%5<KY&MNMC#$91%(^9w%19|6Qe!XIcE%h-?@<{oV^dkyRS2&t8c*h}vvkIdH=jW)d`oQ@eSe#%Fw|y%pGW$WdP1F-QAE&0Ub+VU!I*pfe!sRjZ=j1fvnaIS^*(K2)E+cHMN!Ul9qNW%SOpWHSn|@9ai51w(?u>r9sYbAmpUT9`ZUVR@rT8TpRql82AoQadKIEi)Pshn1TKm0)8ViCas${Vg?#RcRFvah*>u}P3%nKx11;nT}h~Es-W##K~OWj4Csig78PQEb9qc}cAM$a-aHVs@!&+TgO4C7~}wSDM{^@8Ve-HSZ0OED^ruzk1t3s%@E_|;l^pMazu@j`N+d8($|wtQwAcl$t0Wzy%zHh@<Cd?0Msj+09U)$^iDq27;U7r8Xq2-4ZlACpRLm(t`uAr=F*n3L#VHQ|Y|YdPq3l{ZI#Py(KD>i~Fc;p$P}*dk16NBkKrG0Ge~gxO~}0#fxy+Df=jk}@<!nJZX!3X<3z<oduInlu+(s_(lSEW0f%B6#54s4+yLEjdhJjXTmKa+F(trE7+9_%_!N#`s7>e2d)f)*|s&@^sAzgWW~RU^mW2JHU_X42^EF9qK`!N(CqV4bc33@NYth<g$AVQ=hDp#L`*iYZtmrW1DcIAmgARy!^Jo-Jd6qEpZ@NtvN%=q2dT=8=@RdOkgMFl?8Lz*jE1upba!+7E=7RR#K(IK|DF6dm?~b>AX^0i}rG*sF)R*Bsw{^UUqL_DabRxquWDRsnHp6b#hOL>Eit>amf_tt%kT4gFuQpFj_)MC^8@Yxt&oVPQbRCtx#?#V{}n^p>_Ppl@!nbAQnb$J}LJvrfXtEN1Mt|_|bl})~JiWzg*C!&nKraPmI3pGXdIC(F!RYyH<R5i<~p%QFfqyC){%i%#gM=)rp#|kH*>N{vrp-PHGsNN5mp_%H#*vNe)b9nMrgjv*%LX0Jekse%!wOZxHjeX6+pT-dr8z;)~n2nF9c!6ErA8h;Pt)sex_L=*9(9zviyHL&9fM6*tT3uQ>#|KL8lrO}>e&p|>r9n<rCEyqIgJbrLuv&9VA#pe#B+)1xJ!C3caelv8$#N#8t(v;va;u|cn%P^l6Oj3_xqpkG#8f#`ttU6IQfk@z;7y-Qwc!usD1kFxz*St^jwRD&hXpH^kVhd@<A(=sB9g+9pg6hU%hi(ckrGWF>~n*vF6@pbr&A4|8ayzaqV%ugWvUkDrs*=AmKG&jLI<d0*@sS41mom}&L1qD86;yCJ^108;~gQ4X@d5qPDR;ior<DgjE9F!^EnJq#wMj}Dxd30={A+M#L>E+5u0A<}Woi;mg677z^#n67gEwaTu<-W6Sgs2KQl%H211X3Yv@kP3_?>*Z?i~Q8s1-u#;(2eQ?0~h9sq#0YPy(h*0cd1}&`GUWI%M46-`Md>k>yQq-!gh=6utuqCNZ`VZlKMPlgHk5E0Oxblq>t`od!Y1~ei`_@>QWOv^l7*X$tas9^p*128jMHupOwR4j6SeAWX)D=MO%6b6)qRn7(udHOh3*xl~Cb7T?*Jgsx$G0L~9ZhPVeFZkr)WMpc4wF1mw1*%8W@YCbSw1)8Sf1!dKj!!i0(&R(-+MPQejX+?nrgZ0l!e-5k&-OONreG#%lUEL%2mw6n}&?14}73;3^w2uu+!pVio{DMLLj7Zqa?Br+;tAs*V5X3bZ|r2OzflyBo)n*pRCY{6<cd)@O74iFtjr{M#Jh?$wJWwsMDL0{7{k%(w;>+(KHRnD}=Rwow1yMT#M_;emX0Y#Hc%HVd4JsB<%U1Y5+C&H$dVYyb>Q`WH_`;}fz{l_P}0`FAYAuZVnsx*I??FcX9S)P^WI^mwT@oMpnFMvE?QVpL<@XXvngJ<i-9QXGt11&D`pbr{)5%!UYczf)=h=Nkj7NsY-m9Job>I6zh7aOm(N2PH<4Y|Lk(g`6aRIg%`Q<b5B?2oqeuvMjuSJ9@(^Um$5*W%X`dmbX$)=~>2mofiGNtB8l$t4}>pSb0aJ)OZl+GC~y?mi7a)5JHX&pMj@y~PsNDG+g2Z8HXRu_Ddy>SJ)ZM{isYHJr+(3maSTe*v~*$s^CE%{zzXw}WE&=}N1<ACj%c?2=*?LSHIAiOh~Z>I(`z3n7($ic^RY;DNr3CGOE!aeHMPV{r3e6_rBQljHd|Ie!d56%!VhG{~;XtZ2ZYr}II42G9?H0yi2AXK#Y}DWqKBwRn7FOp2u29%azc>l~o8JU;Hf-8PMPP%(k(;G@zFS1RZjLIr(_HfC9pM)$CeAYqCCuNr%9C3KV9WX*8yT|5Vyan+o|5!Bqakh(<E!GigG^lXFT0Pb0mdd<RL?L)~#eDF2AAtz?Sw~@Y5sd7>r#NkHxBn&RF*NgR~h(~#wr&dL<eGi>y<oB3{x-Fkd0%#J0%hj{BrRJ5~w`l}d2Xkg7e1dA0c?vglPMX8nLi4>Kv`{;EqYb&zNbB;6z9#J!^QdH}$r`^*Pd#~Xif9LxQpYCBGzth`FAA@Qw$Yz~V)$=B^`(%)q3Zb15*4SN0X`1?6rcZ7ud%U0wr8n!OMdV5{DUt-?k`DoFq)2sc-|o^3ai~m{X-fCe$CAh8Y{!v`TAt;x@hXS6U=UlJdpDTuw)Vs52dQMtjY;IEnkHk7Bpbjlom4n8f-$BW+OKX5<;pLOK~4hnmvY!@Zthao8~hcj%|e!s6JZy58d`^#ci3lw8BaPN4OdhRYF$MO2|-4VHm~U3NwH(lb1*F36HEu8mk<tS+e&iSiuIK0PNEyy6oP9$J&H#Q<n6z{t*9N`t(yTQZ?o=o%kWU^&!W;wsL4APuUB&)a3n3l{+I;gEu1|L%BG3vznkKMSXt6(0H1zg)m+Ej?N(lZiy}I_=qTw^TeidWJ|N*gv|7HAcKqTdtKN0A!x7RZZ&(Sx=f-gv&%o4e&VL)!sga7izDK|V|_Am0=On7rkR=g7VRT2^9E_|J42p5M}6pn#|~GuE2aA;@vP`G!E*JT?xH$KOY6t44ddoAkGkkzoKv$5c>FW5yC3wdoL=Nj8FIAIR|GUY@)LP*Gda_iIU=609%!}(j>sZ9<sUu8I*=dZx}B2C&rrncsRg@w)y-3tygJnwO!HFWkT@GmrY>baG$vojXWL)bW+?P_8x6-;Vb!mk*;Svzw`4UiW9=F2VPJuRXif#p-uP`b|GMeJbmXy@DsY%2=7~j=)JRbG5_I|}KPpft83Sg2d4z|!C1+7(Ds{sqaO{>|R=`f^^tK;)i_LR&27k!Ddo80GgSwlK6?94UTrNWGmL`ATiW+d#Zqu?b{}j$2wNUEakq761yi1=8Yo*#PWGT3VEU{dI7_T$AIfb;a?&Mu~U(5%3SA=JPK6WgwhZG$=+4g?XN)Al=2XbkNpR<|^lfO6Za$nf0GUu(E5b8fo>+1;~d07!od_^hv52+Jy-c8`>N`NLHnwWL5g@IK+su^ZzL4$JVsW5^u-O~vZoKJLMH><^A>`bboEW`8ZfWgilUVE<PQYBuHIBLDsAkqd7P@Se_kV0<xqA?<!e9c5$V_8JuYPVH)8|zh0AkJm&`B19<HQUfb<hPl`PaS=5&`Dj|>X^3kTMf3;>p)@(Jq<c-ieF5&0l|l8u}d0@RvPJ^%ios=R25a0Mq14({M1DkSlNjpQ{wF7K>JiX`Wm^|Q0*b=m@NlG{1)mrtH&##`ZapgQof^qwJfp-(1h<j)VdE^+;_qFI6JM2m&U{zl+um4XG<8YPb-+6^b#OYn#)hUsqi&?BfeOaJem@^?hXrpUkiq=J3;@Pu#>QTa3ovr5HA(_qmODFPZ0#-PndAss^GJdYrEUAtr1sXPORL0uqS@2=}nXa7UyQW-Q_j`*i=K$>F4nm>QS24jT8PSMPlj|>2KoIA*WkPyAACK?P$ve2sQDiW^0TWr8S1FZk!#$u*)ehY%@=m2K3$F8CwodKIMvtYFX7*Bl8DcE4*^_o@zJLi-I5l)DafK0R~|H>Fn}3EZ(CLy!*nX<gZnQoz@T+|FwoAS{p#EQ9HwLY3!YAr%jW&u(kytYO<rN?Q&1q6+k6@qs1n^1z<d9XBt*gr@k!N@cr4$vWh&>2tv#dx+0T^_<8e#HCDYHK{7r&45;Ra`oQ+_0x_8^-+Uo1TPY+1d_8|SK)xbZjZDNcVy4=%<>kL5Q0w)&xP->N@N6MpGABEG;ms7;z3oC2m+-a;iLXl=Jfm0sX8f3$KIC{i(5{A$3?UTeQfCCIItxE8YC1078k|m0&*GG6brOI!=6$|G_iga4!4q5x(-G>ThszeUk2MQAlK69Xbd^lHH&0+*A3vctl>6D6GXCz*q^vZuNT#K`PRJ*3kqf5m*Rtj0u7sXr8<GzJp$U~6dWQi8C+MY~$})eUBr=zPRL1G-F1}3os!{ldFf4-|m;WyOhK^-_hw>mT4v)r|gUGm%F(S?Jm)(G6MM^;_9hHJ<UpEsceVLmPY|i3wzOIgBt8(S|u3@$O{3%u;#MP8Qam&lG3pdhZ+`-HI@s>m!>p^nBdFmK*tkRmTPW5e;Yx|!s_o5C|%p2gXeqMb0k4A)G_teqC4_3uqrS_HqWTmzi?B0W6Pln>`6EcG&^<D34j3{J^S*DbiIac1W?b4Sv<?v)&P8enAxefut?+B@)!XS&k;#%nk0@Uq9aPBk7WfN*$Babv@;qxjMbwpmGo`xykjjjzQI_dXCfa~#SB7nbh>QZ$7p-ltM3|TBvMUsuAg*AqVWpjU${FoES1xvP`Ks`;_i*d8-rE;xt2kH<_u)wiSTKR3%p+&A}xT<&>>ON|XJ$^NHv`FAFo!cmAY<Xy<rz7y#2z}!>DZUL5&7lBI<u!7=alS%fC#}Ikoa~k2>s0u@yc_R}ngHqlhg=cH>xb%>#niJ1qjgr$gS#$|0q259Yno$ENkE<mrg1Q@RFR@9bg4kAWIv{|cKi9C@PSRb>j*3bT4}$Mvx~5de@U$bWvJ)-*Gp=?p$kz<^<)=j!Oc0LZ^Vm%RSLe-0~HcCyFqjz*X(|s4VBcbmSGz<-{tm<!0MJ{d8nF0d1Q$#n!x-vgrW?>ej{cLA<(arm+pNod&JD*Z5vFgN>DjFr_+iPHPNz+i_1tw@Fx*f;c7S$sc^$mY?OlE8Hu$H!j2}hdZN5Ld&_Ge5`AzMW*c$!R#j@b=Rfdil6x<1bqF(ViWApB!mP7LB+Z<Hdnj=AzTq*1n}9hs)vp@lk?=Gtwn!)JD%}$a2U#_-4ovI{kHy+pOzURycp~~-Ti{Ak*KVitDxwl!4bVE6Wpq;&U9T?j0tA(>%Imk>%l7lt<oMb}i;8<uN=~HkCz+2niC=r^U+YfxP&3;WUYm!kldPz>GC7*aIdVZGNMIxkRuqKdOoU-Wb`1{uMP=FAgWN$==dXmej*QtozvhO2v89KWwA6P5m`C!`7+!gR+q`gi=FAhc6<OH6S1fXnpHkY>tj!5pDnQvR&!89O>i2ESd=&55AqFq!9!4+(lJ#Zb6t<keJ&MR)w8j&J;?W=hUEVPkW2!#_@b2~8U5QsT(M2{tEoXj%G4sHWEdXaN=f1h0f!$Ldm18j0=Qp5Rh$d>m1>kWD>Q1;^DgLdXdsk_U#}ABN2-59-hW5wXa?M2rt6HdGrIRv)VJFeP<5{=p3oqJLo&nniveUr6PqIGev%zW7jlH^49{W_}QnjM{@{B<kHY~;1e;O(TlBH1ib5F1~rNvpByCX2Hh9_c>?RQ%;rG!`cr4Zaz8)}imm46zE{yB*mW1yD>99|>GpDT*-t%Vh;Yi<1&O|JG;@Xk%F)X{EJdsjp^m91sMZqe3Ga6`u&`;Y(s+ka3<5i=0X@Ge+db&6sTP1*WyvW0m~N@aXhWnEO)8W+sG%iFiP?~Yw&0FyAqyA}>%K6){9sBZ<uqMcSdm@vp!d(~68w|HCe1>tgI4>(4j0Nu+6n@x<1+t%pS5z8i9f;78Idufh^J*l<sQ6-aiy$pnl3SoLZ?FZJJQ!6(-O!E|^KoBeSy<I+!Hy&sFNZgV#$4)E#AUN4+nSsf;uy>t>s91{vLn(~r=T`a($-Cd@jGU*jJ6^xQ*RnQoznJnriYF!Ly8z%@+VHt?8)2f>pEB~mc&s{%X1M~@tsn!J=eoXML*12j16m|VuMI}UpdmTlgOm&3h#^&H(KlPb=SXxcXf35pg(d&H$+ZCZ+_Ci|3}4Zgwg~vC;St-2BmVxTBNJbkPEVN2;+&oy=J*{Iw(?t^#`|)fT}&C6^I~K@_Ts+xc&@)$yxd7KaMo^3tdYc)ht?JfgD0Hb;6Z1K{YMXyU9P`nYWk{e|3tVh{Not_tKcJL;v2!}P6KjScyS=UEsO)w+>6>fZ8cZrHEY7`TxmCFz>D%jv#7xm_~fSj4i?gfON-W`vU91fwql``k+V)Q-G}eC)WuWVy_|%E8sG8H!2>}lbq+?&w;w42`lFbU_xHX(bF$meOEyAqaxxFk0(zK5Jdc7u#e5uw0JkfS^?RhARfNF&<tP9HM6QT|irD6016dYgURa-FgT+I_4VVx0(!%}#Srp%a)A*uh*vr|4sS;rTqpVtS_lG3QJ|692a7y8HWm6#66e^kBN2!$t$!(Z~StnVfm<Eg9t!;hp1$zCe51W6s%>1c-^_h@`zt|Dh=gFhOn`cm&6Z~xKCQnRx@qc*TOdLO9L;u2tzZp_SMV@W6+08k$>tzg@M1|{k({6B(VL{UK0)h8$kqQQ*cGOLB1k<E1r|)!{EJq)2+VT>KPvjc)b#qQ<GQAZ+Ju09TLn~KN(}>Kf0xJfjPiqH{6pGlbAE>A$X-T+rd0|u|1)cQ3R$~yt5pol&OY8R*%B<l>-ji*W?1D)73m7}6^c;l-n32PGrp_Lk&t<Yojw$3r-;35-#C_5@f#`;r0PaxV{6l~RYPVN3S%Pw<(otUt5ddYXgG1`bt=5D!t9|rP+x@0FfjLM-SBEQkHju;f6g9LHIntQDM9=ay^p6M}YeM!G-mxxi_(z^REdX68r}{C$F&9MS(4)8@uS}J^08IV4L`q|zXk8-eRaqizOp}_&#or1-^v?)55wVtTM8j^K==oTiUD9AKq;dUYx3G;R%iA*0RoW?@H|w$XzmFXN(?a`Ka{OE|&J<Zv8cDUmAjX1vhkY5yJdR!qrV0;xMgRr2T_!{mWu1V4lS0+~yydmtIMaZ3+6vIgV{zOb423rVKFA<%9XG=5qB4!H#inJ)WPw#k(=1&aI@!@*l%y?W)!(Oi3)We^ZjfLrO&kJ9qO2yE_Mue--IYj-_?lOp)SRAS6ClFrsr><_FPBLsH1xz@At-?>Y@8ObZb6X!nHSa2B|%|kT&Nrn>IkqMiFs})3U@^LQ)PkC)~cyy+O{X8M1Kvc-bQOu4z{i1S$?S+o3og{)G-?9Cq8$on?u}Na!t1|rkiD8r7K`rH3L%|RKCfn>;|v0xT9vOMG)c<p4g2<v=fFN*@>VZrvXwp#=9vGQ}v$4qUDbBboDsx#NIE@A)ro>{R-CgcbpaWqGNA^77XNt^(%8|k7+Y{oMvmGyve@N9P79`D_AM*NxgR7y0OC+u2uq4yxMA&k_+5;c?yki1J576;o7Ml+_6CVj^t?dvS90@+6!T1ajWZPsA9YC4SxL6b#a~LS0)KgjBYS(?lS1hv)#?LDOf2{)@rrS?|}jD?uu8C<zj1JBk)$oWKswu4i#%6C(;+0DSes)1i`4}xd*ehu-KwVqorC6@EJicf!duOBFgWB*~{AfJ86DxK}>cNmE4M=-2`~vTKND!=rRHz#1M}lrctn=1!kY|Bt$7O^qg;ZTd~>RM9Aj61hu;hK5WIgiY1Fj+*36E77@X%Q;vd`<kt&0bDHc=@Qp0P6dF;@kUN01-FeT`a*JJgPng!U$nzsZqeo^V8C>j`dTIlkfGE>>1pR?}$(gz+qnMS#Y;-U*lI1s%<b>92jn4H7x{A~&2|V*r+pPmve?s_ZVUYnVB7)GC(*J=Vblb7TmTqrm&-14gcMh^|c>uPA2=1>4hD#Bt7TeJn#MN1K0e!>_b}6Qv@Z;(SOA6(;$(er3%ho}}ac6?bgJ;0toG3~3(Ed%jh7%inD)Nga?14@&s;<SNR?Ma>)P~*f$mvb%i4UdyseSY7zJ=&9c$TN`udGJbb~unJ>N1)@QCDDbw_OQ<Rr8_wdQ$0sPq^vtO<A3r3t2a#6#N$~UBI>P1qLZg@Wl#xb9Mn;<f-*?Jv!mQaWvS}Eq1O{F<YX*eVn)BT&R6}z9lE;vb*wtR~Ui4isvaP5Fg5;L*cyrw2iBH*HvUn8^lQBA>>?4-OTP*_ii<8)(wYqB}g9YEsj6Mr*S~_Wlt=C#G?(UP8oD{CoPR$b~nFn_>60dE}5qX@5=Mk-7}7#<kv-Cmrib=Pq&o&?@PZsfQWbl7nqqlr19`mHN|qg)Q3iMZ_Y^wS++G24l3Trz4i(&c`SJg+%@lHwpRUMVHHjn${PDHyH0evJ)s4*Pxs>n9sI&CDpxafpxYom@;MBd2T*jKMSYR^FblXd=%JI996!-0qei09cI92tH%$xEnj;N>w-5g~i5qKiX5zEY|96~s1m2}zf9K7CGX+p_T=aOQ$&peFYoxfWn;}E?F}_ZZkrj`Fe+!g#3J4dv33Jv!Ytoe5BdkMs?i$emO?RUu=V`c+cDV-xf-srn9)TC7xgrb{peC!ZAVcs{^<-?~p&c!C+dk|D_s8(tr(RT=A7JPGe=7CO6x`K>$UeXsSmdBM-Oa$-b&jW5B~a&H+2wlfI3Lt(Un1s{u!A47E$Sc5$9v8PL1}AsX3ud_0a;xFv>b&@%Ggztec+1iCSSN_EmMuCqcfq;OsuLLv;h5Y0k+x(7uz=>=Jxmut`K~K{t>DmLdfEttA)`;)l#JS&d97_6kdXa_cQXheqaUJQf;_rp`4Bq2KAl%*bJMUJu=ad=}mvqx`~Q(A@}jEM7pPgdNGu(&$m;30rY(lh>0pR9gd~E&yci^uH|}`5DKjekj8CLB6Jh#rK+EFK8Ee~^)ip1fzyiRr~P0^*E^R~u5#{-9lxcTY9eslq@F&j);FmzLOw+EeMPGf;YT?-rdRhl7TWI23*YE%^xjc{0rK7WLoG*O?4{2*yQ$Tw4SBJFo1eUzocrU6B;Rdb=>{+7Ig@~{Cq-?-QbGY+Urp<xK*1e4#*se1KI>=1=@SUw!R&UrG5T`1i}?M2>4N^81&cPVx2M3x8(=2iYZIH5hU9Ldjr_R~6bIGdp~xfEd~SkAY<qWSqYE`30%kb{uiI*r*j)mLE9Yjm^_TZUW6br$=koRCaGXy4S(gbQ6Fd?#1H0>gWj|DnH}3t(;-=8A8%-q`(5$kbjnsfz6)6l8%Fh3^q(}n;+Z|u)fac_fnf@E0fA+e9Ov*blvb+OsRRHQuHgf{^qOF^tTA|!Pr_f-%a_3Bxd^2nWK~w=FIssqJ*8|Hahc{dwpj!ijZnW<_i+L&rBtpmtO{fsVU#J|GC%ukgrah$#b%pGwn;%yVUSCTE5sYiUtze`SL~#z`U0d`gZx1Ks?`t7e_Ff0_&iZ{RClu5&D5Yusj0G6<W>kf9g@=(;hVwMcRy)K5nZL^ow>4iF6id)#Ye2?+sJeye%(qF4%NrJ8tGnP*Pc=0xn_4SHvKp)>4_m`M%|MX%qYjV9JR?c8je_5RtqnbM8Sg+CDGJ!ZP$|D~8o%5WKiWbg42Ag8bJ1LSWY*%1?rbSt|7Z4V!qU^T-t`IxjYq!6hF>twVPxU8Wwip{`~OvwGsH|`vyZ@QvdNpz*lJz$6sSf)q}aJUFc_?V=w1Xx<6>nR=*rmdsVe54j8uWCmTRNtcSwIRndygH)4-VOjS4$PFX4cDBx8+sWt+%y2m(JVVU-1}%m~|T%80-EVM_(`CxRuT_*02}hQ$lANsAQEQ-E7*iCKi`d-|~fDzrn*x3gN4x%kzkqIWn3+FzM1TNuh6amgId`9OHw2}3wRQ244B_U}~#h^XPpxo9COx`cI!sMYx*&0J>|Df5hXdwpvhY+=dtsC;bNJ^XNi!sB_ZUZ<2q0;kai2T-GT%P%|Qx5wd=T;69mg*9|kr@Iq=gQJmhB*0lx-TzW2NI<>|(?)$yO+YPSnwT?NhRy`0-exgyo4jt&^>0^NytLSbUxkg+=Y$Mt1q#yOocSFsB1j<s(x5jCI}q1p?RESEYqY;0$YM;^f#eZ7RE2=rE}Y`L3RA&#CH_5dPPwAdA59000|pxmbKrxM+n<fDqn&aRU`bD)lzI*-hUu738?#;|T8YTF-|9)}RnK~2(ah6rdMVBohB|k;-tQ}5sPW}d`zl)gr*XwHkcxBEiFiF$tKemV&M(*krjiXa7LOfGqa{?uG=yoW#Yt~(=D?+Tn@Gf%K|AD1XQE0q<TXp3(^PW57G+++NI4XZPoof5c@_b;K1m3=Jnb|`rp}KJeHJeq8RmuJpX-&AariaAACwjLgq1Ki6j!J}#<HZQ`-e#cBQ4QBn4}%yrFY6iA%>BKRAlvrn#Z4GbaFR((NO68f7BV<r46oXUa$9pS)xkJOtK6a=?K!$*MxT#Ox6d1%NMeH0A+Vtq~O%RgNcow3sw3~$BJ`cs@^1DB>;!qixOOb20qVzJM$<heGm5@PJ-1PqJk(}0^7F^*3k{dDfkQ%EFPTEqZ}{F(8;<X+^^$NQ+HXQ?UC_F)EVpTq!=}Qz|{Mpm_+9~$>2EJH<?$c16cBtK?OF*sNWJx8>eil#;$V3m1WhInh^5zM%?hJ4^a-RwKTTp(-Ix{I>|#^T#Mq}wG)F-<*a;<+*OdxWY9Qw+2YgP#NCpDkCEI`1_Fs_(<cJPF00&qLZ<7%R9_I5x@|LMt|boSFycxv27F9C3u`QqT1&f-eUM6Iz|9YA(fj1%fUwVTyj{T^JB=<2Xr*C76DCzANgL~}$kZ1|GFyBKs>XoQr$smFfcp;~6RAbj_`l#+gQ0Oy?eGRT`PEey3GFNnaV3`SW&gY!ph{XLy&EUC#;ANEF(ET>ZYG!Jr70wtpN_@jdMM3x0OFpUd1uidbyP_mIrms)^WEwO-oYdZM%IK@Rzwk9hPqjdx(>=jxY)-6)yJjJE58DCNNl~+>J_A;>!Z{`vrB>Q<2^eRc@V+W2j!4l4|w&|Xih`{^F)bXOVVu02x*-E@(PH8)Gj<oma@GK{H;@_WVuXk(4$&rK?BDv%tF(lSAY3_R+HsEu7FXiM~t)9<MEiZ)Yy?g#>w?5m`A(;vR{t0FtQ3@{1r!`fo|?v7O>Ks=_=%VBu~PjbByL$PzD|HqN~8V9=L}PQWAMlDl$SUnvqev)IGK!5YD9UI1_?Cq5Em&6>{MUze;uTVWok6w|>>-<8y?N%puIT{x&+2!7P3m+Cc-<k(qCbmRe++C0Sjl@S%0}-A0Faf&xKgQlBNbp-j04CF*4R60RI~AXCF_w0}Y<4aJVx!wDDl{lh)tEm>0L9YUBgxphwe35Vd=sf>w{n;lRO`NCsJg>!pC1f{;>xlSYbR!Oiq9ZKZ)pH(J3k5b1S7xVnK)S<3BoB>h%eS?E`@Jf+O*&*Al`^qqami02!(<NA)kh5?biGusTEZJ3@%I?T*!g7EU+&}R^xRfeoYK^A}VR}3Eppx#zT0Z@Nu5y<>JhVonZF0knERF)>K@^=N&MtQY1+NlL!(ls#+)En%18`Wx>lih1D)SF%!)c8|D^Th`gs8@#!pMy|gAD@^G#0yAT`PKtXS_`{f~`Wc)axT|U}!~D%!4Yd-mSWqYdyer<Yct}54F=b=uvbjSgYiaUzV(B)-H&`UB7waOM}OAEHIn-uQ{6&kEf|<S~hDVX3#wp4y6B|m!;rw97;5Pb#HE$F4_5`$g$}2Up7pRSFw-mqN!~Tnm9pqY-DL0K*DP*Lj=#u09%TF*gW045#P8%Tyt873me@<w86^(-S|Ds6Sdvv2ih%l1CFj1<v8*Et@n)Y*YB}$twR7DqVP!x0??BMEcIvlcNLu93wF7qxd^+r5|KQK#cFNGY}9Agn9&7#!WVvUMKz65bEk&jDn*D#7l^<Cy7;^5CKk-WwbgwdYxhV|Wi>UV4qTj{cP^tnPN0-O4U@(ZE={B)nSt2`xdAVAbtC|l9?q<oKB{cN^-pFhz86K!e*_TLz<!*(tU;i^nnUEj?gV@QDkjzGpQJ3V3JDU<1$w|v6Uq=jO%z^mxeYBdV9`Xow&Qi=qQFgivOCbmD_vF#m9yZ)2CRuWl~Xok;k|*RRsBhsZ%bI(IXbSS7US1GRJymbhbMT%AAVVCTa?|N$>T~9B<&SF!J2pkTvH~TDKT_U2vCbDOIXgxl(ns5MmG#Z2iSj-u#Hd1W)>N53lQG~htPo3bjCN{OU?D@F}i4)fNXln#dVV<;RK9}XTz}HMw^>|K>OkZ<+J{=+B8U*89JTGZc-)&@I`axIx1`UN}(iPGI}yA`c!Snp0+z+OcAjIwB8tV2MP^^^A3gXM^fv?6@7&%-=l=)JWbJSP+#)vK87h}_6-|ByH>!^qS|3!{aNeYAQ6x)1pjb=C)ySB-Qi8Et#y3OxBf9aN;8k7x@X)d0i#H!CAk%{T-{qcJBz$~G`vEtvhZt-sVbpT!6J1HG)uRe+I)M(y$`+wav7=f64K6uc$=Qpg^aU4R_=FQ*|t|<P0huKozAts50WJ~MBEvveUL8i)1k4p${WMec0jnqjpbwJY<&5=1}-R0NiK-k+8~a99ohgaWVTo@zIVwD36uYA92qh-kTx@S{~qx*1TAxHyC!K@5*Fke5Q{Dm=3G&-ll_B?z4k*&qJPP>H_A4VRuElMldPhIjyAYK4Z|U6$L@Q6!x%Dd6n$vQQ4F{AHUMN(3UC>$-w)Ijqg#!4U>01n9+E3eLN4a#Kz&I%#izL6ISgzJzCI1Gbl7aUz0g+d8IL7+xv=-?G!GS9z8JvxXE+&Q5dX=y@OqYdHTufTo!@KEpD8+J@1_BZki$(u0dH#AWo84(9`<vY3S$08g{ygE`P?sZ<OuMZ0t8RLrR!R4pVQkK6`3y>BiS%8!8u7T#sK*7iY(=b9|3;KfP}pMv8W#tskuy`YhL6LXszHxthKy?V`)##cvQn8mU`Ka#G}{nmVyHLJmPD$tkrQFso;YEh}7)h)-kmQXeOoBbQO2VaMF^eeMfhI)MS%8ynu4Adib(}w{jl&o(O7FVQ@gB)u(N#>RmUOH=pkdPo}OH23<Mh04umR7<|Al*z`jQCFho)VVN)db`hYuA(=aC&10<7l#2s&I8F_g4nZBRbke9X)L0AH7b0Bj7Qv3e%A|end0`l%Dk2OZH0gy=dir*(B63Zrvo$soU=X;1t4>j;0y&(|@~5Li0x5hDI6&20j}g<}Mbnfk+rq3^z4xU#jU0@KF>TYlO`+iFX!;Ck1V~4ruznOnfr>E(Eomrl|2&E`d(>fF!Q+QwM9h4*=LSNpa;bET<inwxk@ZJig@5Kc-)9BiI{}GSB=({AT#V#yIT?D)X2?q+$F6sT21!(O6Htc*rq;xV_ew*i)d}T@xOuc|>>G7FM58X;$j*v?an-vNVxKUPNOvcT?0xO482W!ya7LDgUI$#Gdj(tM7>PV#$IVoW^q>Z-@hBdBe-z%PibU~7we#n;M_SwW#$2MFfd^9LAfITR#FuJ^X%qqrAsc>tsc#K0wtyU!<h^Dj<YbsgYPB?_Qygu77)UB`x<sj{`a(JyQUaJ08J*4po#SJYFEgRQO%nh(iSpzIFwD1Z-*xp?H~{dp^`zvBow<Ah`qQ?8^kno({Y-1yVRUB>fJooi_};?dUowx?I-ak$!FND-J1#<t*M2ISW?mNWP!9!dWtsyZ2-}e;jFo_;2w>IL72?=|f0tl`c#a95dCv@?ft-$|T6kUNWGg2281;V9&1x{R=nR9+0)zyi4fAG*mld60vwMwT&(Om#2pBYKL9cG{$qDFE+Q3{7DU-hPuWnVoQbq@sZ`37EkM+z@T31J{o?<)=ZU5>R$o&iyrAVus(nTsRoD_NZ-}Am!F{CB!-pzw8o*(bi3Yn>?bH`EZ>VS}W;T3q)fR4MyvB$ruEscIsNX;9`GSb97O4xmO08D;NZ08kGPpMuX9Fagh=Y~P?@?#N2lQGeY)u=7OISNZhhFy(3Frs7dRQ{1{ca8vpMH+(mBE*=94aG?^-KC!Q7dz$qdf;X1b-q07`L@lITnM|bviD%ivprYers)ZZw>^<8G6W^L`G;~1c||3M6et{kPgH9+*4^boA;g2G!e;}plxW({1{ohVD~U)Z60tE)1@pF(!6NKb=>H+O19p^|NBHuSV#Ax^6S^Az&0)g-GhV5P=)dB)YE9vegTCi`J}#Gk-I=Ty2_>+waZj=u)eWh>hhE#QhH6xe=!Dy~*ZD7^0z*8emS@ax=K?P*W!)sE2-v<zsEimd$4i@fQqK#8sGn$G(X2ah-7$OudY^r-2}|^5gMnv@ZnMklEw0_#pmyV%3Pt#dkqPnAu@$1nB1l6Y$gxcrRB!!p-gMhr`Z)&Upc+G~_JK#!$;hG{UJ~f}jZVrVBQil3L0lZEC~u&cQ<73y#3@K*$ULP1RuH<ZJFJik6@Kg}kWji;cl};A1zjdTU&CPN!<+R4zJaVq5co3kZYz9gu=gRf$xu4I7|dvqUNqAW{LM;k;sG_y$afSTqZZW#1lYByzFzu&EW2s?ND*$X;G=5>Z5|DuQQ55JUKQ7l!t8dv@<<pQIKz7WE3=b|E}3b9vOcP^>ddjeey%1rHH(E~D=|ZDx%jMWsq}s^v<SLIhzFg5GBz$jT1#8R!wQ>EQ><@VhV$2AbprI;`Ynv<tdtg*4d?K8K4PthT0|L)00000BDRUI6oSAF00Gg$0jBQ<+ngz@vBYQl0ssI200dcD",
}
for stl, b85 in EXTRA_STLS.items():
  (ASSETS_DIR / stl).write_bytes(lzma.decompress(base64.b85decode(b85)))

print(
  f"Bootstrapped so101-mjlab in {REPO_DIR} with {len(list(ASSETS_DIR.glob('*.stl')))} STL meshes."
)

In [ ]:
# 2. Install so101-mjlab in editable mode and verify task registration
!uv pip install --system --no-deps -e /content/so101-mjlab
!python -c "import so101_mjlab; from mjlab.tasks.registry import list_tasks; print([t for t in list_tasks() if 'SO101' in t])"

In [ ]:
# 3. Configure Weights & Biases (offline by default; uncomment `wandb login` if desired)
import os

os.environ["WANDB_MODE"] = "offline"
# !wandb login

In [ ]:
# 4. Train Mjlab-Lift-Cube-SO101 on GPU (2048 parallel environments)
# Adjust --agent.max-iterations (e.g., 1000 for a quick ~5-8 min run on T4/L4, or 2500 for full convergence)
!MUJOCO_GL=egl train Mjlab-Lift-Cube-SO101 \
  --env.scene.num-envs 2048 \
  --agent.max-iterations 1000 \
  --log-root /content/logs/rsl_rl

In [ ]:
# 5. Locate the latest trained checkpoint and exported ONNX policy
import shutil
from pathlib import Path

log_root = Path("/content/logs/rsl_rl/so101_lift_cube")
onnx_files = sorted(log_root.rglob("*.onnx"), key=lambda p: p.stat().st_mtime)
pt_files = sorted(log_root.rglob("model_*.pt"), key=lambda p: p.stat().st_mtime)

assert pt_files, "No model_*.pt checkpoints found!"
latest_pt = pt_files[-1]
print("Latest PyTorch checkpoint:", latest_pt)

if onnx_files:
  latest_onnx = onnx_files[-1]
  out_onnx = Path("/content/so101_lift_cube_policy.onnx")
  shutil.copy2(latest_onnx, out_onnx)
  print("Ready for download:", out_onnx)

In [ ]:
# 6. Record a rollout video of the trained policy in simulation and display inline
import mediapy as media
import mujoco
import onnxruntime as ort
import torch
from mjlab.envs import ManagerBasedRlEnv
from mjlab.tasks.registry import load_env_cfg

import so101_mjlab  # noqa: F401

cfg = load_env_cfg("Mjlab-Lift-Cube-SO101", play=True)
cfg.scene.num_envs = 1
env = ManagerBasedRlEnv(
  cfg=cfg, device="cuda:0" if torch.cuda.is_available() else "cpu"
)
obs, _ = env.reset()

sess = ort.InferenceSession(
  "/content/so101_lift_cube_policy.onnx", providers=["CPUExecutionProvider"]
)
in_name = sess.get_inputs()[0].name

mj_model = env.sim.mj_model
mj_data = env.sim.mj_data
vopt = mujoco.MjvOption()
vopt.geomgroup[:] = [1, 1, 1, 0, 0, 0]
renderer = mujoco.Renderer(mj_model, height=480, width=640)
cam = mujoco.MjvCamera()
mujoco.mjv_defaultFreeCamera(mj_model, cam)
cam.lookat[:] = [0.14, 0.0, 0.09]
cam.distance = 0.48
cam.azimuth = 135.0
cam.elevation = -22.0

frames = []
for _ in range(200):
  actor_obs = obs["actor"].detach().cpu().numpy()
  action_np = sess.run(None, {in_name: actor_obs})[0]
  action_tensor = torch.as_tensor(action_np, device=env.device, dtype=torch.float32)
  obs, _, _, _, _ = env.step(action_tensor)
  mj_data.qpos[:] = env.sim.data.qpos[0].detach().cpu().numpy()
  mj_data.qvel[:] = env.sim.data.qvel[0].detach().cpu().numpy()
  mujoco.mj_forward(mj_model, mj_data)
  renderer.update_scene(mj_data, camera=cam, scene_option=vopt)
  frames.append(renderer.render())

renderer.close()
env.close()
media.show_video(frames, fps=50)

In [ ]:
# 7. Download the trained ONNX policy & PyTorch checkpoint to your Mac
from google.colab import files

files.download("/content/so101_lift_cube_policy.onnx")
files.download(str(latest_pt))